In [29]:
from pathlib import Path
import pickle
import json
import pandas as pd
import numpy as np
from collections import Counter

DATA_DIR = Path(r"D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL")

OUTPUT_DIR = DATA_DIR / "processed_traces"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

Data dir: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL
Output dir: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces


In [30]:
def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)


pkl_files = sorted(DATA_DIR.glob("*.pkl"))

print("Total pkl files:", len(pkl_files))
print("First few files:")
for p in pkl_files[:5]:
    print(" -", p.name)

Total pkl files: 161
First few files:
 - ex_bckp1-A-DC_sim1.pkl
 - ex_bckp1-B-DC_sim1.pkl
 - ex_bckp1-B-DI_sim1.pkl
 - ex_bckp2-A-DI_sim1.pkl
 - ex_bckp2-B-DC_sim1.pkl


In [31]:
meta_records = []

for p in pkl_files:
    try:
        obj = load_pkl(p)

        meta_records.append({
            "file": p.name,
            "path": str(p),
            "group": obj.get("group"),
            "subgroup": obj.get("subgroup"),
            "session_code": obj.get("session_code"),
            "task": obj.get("task"),
            "logs_type": type(obj.get("logs")).__name__,
            "n_logs": len(obj.get("logs", [])) if hasattr(obj.get("logs", []), "__len__") else None,
        })

    except Exception as e:
        meta_records.append({
            "file": p.name,
            "path": str(p),
            "group": None,
            "subgroup": None,
            "session_code": None,
            "task": None,
            "logs_type": "ERROR",
            "n_logs": None,
            "error": repr(e),
        })

meta_df = pd.DataFrame(meta_records)

display(meta_df.head())
display(meta_df["group"].value_counts(dropna=False))

group_b_meta_df = meta_df[meta_df["group"] == "B"].copy()

print("Number of Group B pkl files:", len(group_b_meta_df))
display(group_b_meta_df.head())

,file,path,group,subgroup,session_code,task,logs_type,n_logs
0,ex_bckp1-A-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-A-DC_sim1.pkl,A,DC,ex_bckp1-A-DC,sim1_task,list,1346
1,ex_bckp1-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,list,1546
2,ex_bckp1-B-DI_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-B-DI_sim1.pkl,B,DI,ex_bckp1-B-DI,sim1_task,list,398
3,ex_bckp2-A-DI_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp2-A-DI_sim1.pkl,A,DI,ex_bckp2-A-DI,sim1_task,list,1678
4,ex_bckp2-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp2-B-DC_sim1.pkl,B,DC,ex_bckp2-B-DC,sim1_task,list,1563


group
B    81
A    80
Name: count, dtype: int64

Number of Group B pkl files: 81


,file,path,group,subgroup,session_code,task,logs_type,n_logs
1,ex_bckp1-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,list,1546
2,ex_bckp1-B-DI_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-B-DI_sim1.pkl,B,DI,ex_bckp1-B-DI,sim1_task,list,398
4,ex_bckp2-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp2-B-DC_sim1.pkl,B,DC,ex_bckp2-B-DC,sim1_task,list,1563
6,ex_bckp3-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp3-B-DC_sim1.pkl,B,DC,ex_bckp3-B-DC,sim1_task,list,808
7,ex_bckp3-B-DI_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp3-B-DI_sim1.pkl,B,DI,ex_bckp3-B-DI,sim1_task,list,1002


In [32]:
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def get_direction(old_value, new_value):
    old_f = safe_float(old_value)
    new_f = safe_float(new_value)

    if old_f is None or new_f is None:
        return None

    if new_f > old_f:
        return "increase"
    elif new_f < old_f:
        return "decrease"
    else:
        return "same"


def make_directional_label_raw(action, old_value, new_value):
    """
    Raw directional label without mapping to our model env.
    Example:
    action='concentration', old=0.1, new=0.2 -> increase_concentration
    action='click_start' -> click_start
    action='end' -> end
    """

    direction = get_direction(old_value, new_value)

    if direction in {"increase", "decrease", "same"}:
        return f"{direction}_{action}"

    return action


def extract_raw_traces_from_logs(logs):
    raw_trace = []
    directional_trace_raw = []
    event_rows = []

    for event_pos, entry in enumerate(logs):
        if not isinstance(entry, dict):
            continue

        raw_action = entry.get("action")
        old_value = entry.get("old_value")
        new_value = entry.get("new_value")
        direction = get_direction(old_value, new_value)
        directional_label_raw = make_directional_label_raw(raw_action, old_value, new_value)

        raw_trace.append(raw_action)
        directional_trace_raw.append(directional_label_raw)

        event_rows.append({
            "event_pos": event_pos,
            "index": entry.get("index"),
            "timestamp": entry.get("timestamp"),
            "raw_action": raw_action,
            "old_value": old_value,
            "new_value": new_value,
            "direction": direction,
            "directional_label_raw": directional_label_raw,
            "has_old_new_value": old_value is not None and new_value is not None,
            "original_entry_json": json.dumps(entry, ensure_ascii=False, default=str),
        })

    return raw_trace, directional_trace_raw, event_rows

In [33]:
trace_records = []
event_records = []

for _, row in group_b_meta_df.iterrows():
    p = Path(row["path"])
    obj = load_pkl(p)
    logs = obj.get("logs", [])

    raw_trace, directional_trace_raw, event_rows = extract_raw_traces_from_logs(logs)

    raw_action_counts = Counter(raw_trace)
    directional_action_counts = Counter(directional_trace_raw)

    unique_raw_actions = sorted(set(a for a in raw_trace if a is not None))
    unique_directional_actions = sorted(set(a for a in directional_trace_raw if a is not None))

    trace_records.append({
        "file": p.name,
        "path": str(p),
        "group": obj.get("group"),
        "subgroup": obj.get("subgroup"),
        "session_code": obj.get("session_code"),
        "task": obj.get("task"),
        "n_logs": len(logs),
        "n_raw_events": len(raw_trace),
        "n_unique_raw_actions": len(unique_raw_actions),
        "unique_raw_actions": unique_raw_actions,
        "unique_directional_actions_raw": unique_directional_actions,
        "raw_trace": raw_trace,
        "directional_trace_raw": directional_trace_raw,
        "raw_action_counts": dict(raw_action_counts),
        "directional_action_counts_raw": dict(directional_action_counts),
        "first_50_raw_actions": raw_trace[:50],
        "first_50_directional_actions_raw": directional_trace_raw[:50],
    })

    for event_row in event_rows:
        event_row.update({
            "file": p.name,
            "group": obj.get("group"),
            "subgroup": obj.get("subgroup"),
            "session_code": obj.get("session_code"),
            "task": obj.get("task"),
        })
        event_records.append(event_row)

trace_df = pd.DataFrame(trace_records)
event_df = pd.DataFrame(event_records)

print("Number of Group B traces:", len(trace_df))
print("Number of Group B raw events:", len(event_df))

display(trace_df.head())
display(event_df.head())

Number of Group B traces: 81
Number of Group B raw events: 121795


,file,path,group,subgroup,session_code,task,n_logs,n_raw_events,n_unique_raw_actions,unique_raw_actions,unique_directional_actions_raw,raw_trace,directional_trace_raw,raw_action_counts,directional_action_counts_raw,first_50_raw_actions,first_50_directional_actions_raw
0,ex_bckp1-B-DC_sim1.pkl,D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,1546,1546,15,"[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, width, xaxis, yaxis]","[click_end, click_start, click_sum, decrease_concentration, decrease_wavelength, decrease_width, delete_point_from_table, end, increase_concentration, increase_wavelength, increase_width, open_solution_menu, plot, record, remove_from_graph, solution, xaxis, yaxis]","[click_start, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, concentration, ...]","[click_start, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease

,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,has_old_new_value,original_entry_json,file,group,subgroup,session_code,task
0,0,0.0,1680177745271,click_start,None,None,NaN,click_start,False,"{""index"": 0, ""action"": ""click_start"", ""timestamp"": 1680177745271}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task
1,1,1.0,1680177745518,concentration,0.1,0.105,increase,increase_concentration,True,"{""index"": 1, ""action"": ""concentration"", ""timestamp"": 1680177745518, ""old_value"": 0.1, ""new_value"": 0.105, ""old_abs"": 0.51, ""new_abs"": 0.53}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task
2,2,2.0,1680177745549,concentration,0.105,0.11,increase,increase_concentration,True,"{""index"": 2, ""action"": ""concentration"", ""timestamp"": 1680177745549, ""old_value"": 0.105, ""new_value"": 0.11, ""old_abs"": 0.53, ""new_abs"": 0.56}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task
3,3,3.0,1680177745575,concentration,0.11,0.115,increase,increase_concentration,True,"{""index"": 3, ""action"": ""concentration"", ""timestamp"": 1680177745575, ""old_value"": 0.11, ""new_value"": 0.115, ""old_abs"": 0.56, ""new_abs"": 0.58}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task
4,4,4.0,1680177745605,concentration,0.115,0.12,increase,increase_concentration,True,"{""index"": 4, ""action"": ""concentration"", ""timestamp"": 1680177745605, ""old_value"": 0.115, ""new_value"": 0.12, ""old_abs"": 0.58, ""new_abs"": 0.61}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task


In [34]:
all_raw_action_counts = Counter()

for trace in trace_df["raw_trace"]:
    all_raw_action_counts.update(trace)

raw_action_count_df = pd.DataFrame(
    all_raw_action_counts.items(),
    columns=["raw_action", "count"]
).sort_values("count", ascending=False)

display(raw_action_count_df)

,raw_action,count
13,wavelength,35876
12,width,28894
1,concentration,25025
2,click_end,8204
0,click_start,8168
3,click_sum,7769
15,ruler,2935
8,record,1186
4,open_solution_menu,871
5,solution,680


In [35]:
all_directional_counts = Counter()

for trace in trace_df["directional_trace_raw"]:
    all_directional_counts.update(trace)

directional_action_count_df = pd.DataFrame(
    all_directional_counts.items(),
    columns=["directional_action_raw", "count"]
).sort_values("count", ascending=False)

display(directional_action_count_df)

,directional_action_raw,count
15,increase_wavelength,19048
16,decrease_wavelength,16828
13,increase_width,15267
1,increase_concentration,14999
14,decrease_width,13627
2,decrease_concentration,10026
3,click_end,8204
0,click_start,8168
4,click_sum,7769
18,ruler,2935


In [36]:
pd.set_option("display.max_colwidth", None)

trace_overview_df = trace_df[
    [
        "file",
        "session_code",
        "subgroup",
        "task",
        "n_logs",
        "n_unique_raw_actions",
        "unique_raw_actions",
        "first_50_directional_actions_raw",
    ]
].copy()

display(trace_overview_df)

,file,session_code,subgroup,task,n_logs,n_unique_raw_actions,unique_raw_actions,first_50_directional_actions_raw
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,1546,15,"[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, width, xaxis, yaxis]","[click_start, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration]"
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,398,13,"[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, plot, record, solution, wavelength, xaxis, yaxis]","[record, click_start, click_end, open_solution_menu, click_sum, yaxis, xaxis, click_start, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_concentration, increase_concentration]"
2,ex_bckp2-B-DC_sim1.pkl,ex_bckp2-B-DC,DC,sim1_task,1563,12,"[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, record, ruler, solution, wavelength, width]","[click_start, click_end, click_sum, click_start, click_end, click_sum, click_start, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler, ruler]"
3,ex_bckp3-B-DC_sim1.pkl,ex_bckp3-B-DC,DC,sim1_task,808,14,"[click_end, click_start, click_sum, concentration, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, xaxis, xaxis_scale, yaxis]","[click_start, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, decrease_wavelength, 

In [37]:
# 1. event-level table: 每一行是一个 log event
event_output_path = OUTPUT_DIR / "groupB_raw_event_table.csv"
event_df.to_csv(event_output_path, index=False, encoding="utf-8-sig")

# 2. overview table: 每一行是一个 pkl/session
overview_output_path = OUTPUT_DIR / "groupB_raw_trace_overview.csv"
trace_overview_df.to_csv(overview_output_path, index=False, encoding="utf-8-sig")

# 3. action counts
raw_counts_output_path = OUTPUT_DIR / "groupB_raw_action_counts.csv"
raw_action_count_df.to_csv(raw_counts_output_path, index=False, encoding="utf-8-sig")

directional_counts_output_path = OUTPUT_DIR / "groupB_directional_action_counts_raw.csv"
directional_action_count_df.to_csv(directional_counts_output_path, index=False, encoding="utf-8-sig")

# 4. full session traces as JSONL
jsonl_output_path = OUTPUT_DIR / "groupB_raw_session_traces.jsonl"

with open(jsonl_output_path, "w", encoding="utf-8") as f:
    for record in trace_records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

print("Saved files:")
print(event_output_path)
print(overview_output_path)
print(raw_counts_output_path)
print(directional_counts_output_path)
print(jsonl_output_path)

Saved files:
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_event_table.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_trace_overview.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_action_counts.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_directional_action_counts_raw.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_session_traces.jsonl


In [38]:
loaded_records = []

with open(jsonl_output_path, "r", encoding="utf-8") as f:
    for line in f:
        loaded_records.append(json.loads(line))

print("Loaded records:", len(loaded_records))
print("First trace file:", loaded_records[0]["file"])
print("First 30 raw actions:")
print(loaded_records[0]["raw_trace"][:30])
print("First 30 directional actions:")
print(loaded_records[0]["directional_trace_raw"][:30])

Loaded records: 81
First trace file: ex_bckp1-B-DC_sim1.pkl
First 30 raw actions:
['click_start', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration', 'concentration']
First 30 directional actions:
['click_start', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration', 'increase_concentration',

In [39]:
pd.set_option("display.max_colwidth", None)

def action_flags(raw_actions):
    s = set(raw_actions)
    return {
        "has_wavelength": "wavelength" in s,
        "has_width": "width" in s,
        "has_concentration": "concentration" in s,
        "has_solution": "solution" in s,
        "has_ruler": "ruler" in s,
        "has_record": "record" in s,
        "has_plot": "plot" in s,
    }

flag_records = []

for _, row in trace_df.iterrows():
    flags = action_flags(row["raw_trace"])

    numeric_actions = sorted({
        a.replace("increase_", "").replace("decrease_", "")
        for a in row["directional_trace_raw"]
        if a.startswith("increase_") or a.startswith("decrease_")
    })

    flag_records.append({
        "file": row["file"],
        "session_code": row["session_code"],
        "subgroup": row["subgroup"],
        "task": row["task"],
        "n_logs": row["n_logs"],
        "numeric_actions": numeric_actions,
        "unique_raw_actions": row["unique_raw_actions"],
        **flags,
    })

flag_df = pd.DataFrame(flag_records)

display(flag_df)

,file,session_code,subgroup,task,n_logs,numeric_actions,unique_raw_actions,has_wavelength,has_width,has_concentration,has_solution,has_ruler,has_record,has_plot
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,1546,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, width, xaxis, yaxis]",True,True,True,True,False,True,True
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,398,"[concentration, wavelength]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, plot, record, solution, wavelength, xaxis, yaxis]",True,False,True,True,False,True,True
2,ex_bckp2-B-DC_sim1.pkl,ex_bckp2-B-DC,DC,sim1_task,1563,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, record, ruler, solution, wavelength, width]",True,True,True,True,True,True,False
3,ex_bckp3-B-DC_sim1.pkl,ex_bckp3-B-DC,DC,sim1_task,808,"[concentration, wavelength]","[click_end, click_start, click_sum, concentration, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, xaxis, xaxis_scale, yaxis]",True,False,True,True,False,True,True
4,ex_bckp3-B-DI_sim1.pkl,ex_bckp3-B-DI,DI,sim1_task,1002,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, end, open_solution_menu, plot, record, remove_from_graph, ruler, solution, wavelength, width, xaxis, yaxis, yaxis_scale]",True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,k1-xuydcgqp_sim1.pkl,k1-xuydcgqp,DI,sim1_task,1338,"[concentration, wavelength]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, open_solution_menu, record, solution, wavelength, xaxis, yaxis, yaxis_scale]",True,False,True,True,False,True,False
77,k1-yjmnv8bt_sim1.pkl,k1-yjmnv8bt,DC,sim1_task,2852,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, move_down_point, open_solution_menu, plot, record, ruler, solution, wavelength, width, xaxis, xaxis_scale, yaxis, yaxis_scale]",True,True,True,True,True,True,True
78,k1-z9itzaam_sim1.pkl,k1-z9itzaam,DI,sim1_task,1841,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, delete_point_from_table, end, move_down_point, move_up_point, open_solution_menu, plot, record, remove_from_graph, ruler, solution, wavelength, width, xaxis, yaxis, yaxis_scale]",True,True,True,True,True,True,True
79,k1-zvm6hs9n_sim1.pkl,k1-zvm6hs9n,DC,sim1_task,1219,"[concentration, wavelength, width]","[click_end, click_start, click_sum, concentration, end, open_solution_menu, plot, record, remove_from_graph, solution, wavelength, width, xaxis, yaxis]",True,True,True,True,False,True,True


In [40]:
flag_df["numeric_signature"] = flag_df["numeric_actions"].apply(
    lambda xs: "+".join(xs)
)

signature_df = (
    flag_df
    .groupby(["numeric_signature", "subgroup"])
    .size()
    .reset_index(name="n_traces")
    .sort_values("n_traces", ascending=False)
)

display(signature_df)

,numeric_signature,subgroup,n_traces
3,concentration+wavelength+width,DC,33
4,concentration+wavelength+width,DI,30
2,concentration+wavelength,DI,10
1,concentration+wavelength,DC,6
0,concentration,DC,1
5,concentration+width,DI,1


In [41]:
import json

def show_event_samples(action_name, n=20):
    sub = event_df[event_df["raw_action"] == action_name].copy()

    print("action:", action_name)
    print("n events:", len(sub))
    display(sub[[
        "file",
        "session_code",
        "subgroup",
        "event_pos",
        "index",
        "timestamp",
        "raw_action",
        "old_value",
        "new_value",
        "direction",
        "directional_label_raw",
        "original_entry_json",
    ]].head(n))

show_event_samples("concentration", n=20)
show_event_samples("solution", n=20)
show_event_samples("width", n=10)
show_event_samples("wavelength", n=10)

action: concentration
n events: 25025


,file,session_code,subgroup,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,original_entry_json
1,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,1,1.0,1680177745518,concentration,0.1,0.105,increase,increase_concentration,"{""index"": 1, ""action"": ""concentration"", ""timestamp"": 1680177745518, ""old_value"": 0.1, ""new_value"": 0.105, ""old_abs"": 0.51, ""new_abs"": 0.53}"
2,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,2,2.0,1680177745549,concentration,0.105,0.11,increase,increase_concentration,"{""index"": 2, ""action"": ""concentration"", ""timestamp"": 1680177745549, ""old_value"": 0.105, ""new_value"": 0.11, ""old_abs"": 0.53, ""new_abs"": 0.56}"
3,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,3,3.0,1680177745575,concentration,0.11,0.115,increase,increase_concentration,"{""index"": 3, ""action"": ""concentration"", ""timestamp"": 1680177745575, ""old_value"": 0.11, ""new_value"": 0.115, ""old_abs"": 0.56, ""new_abs"": 0.58}"
4,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,4,4.0,1680177745605,concentration,0.115,0.12,increase,increase_concentration,"{""index"": 4, ""action"": ""concentration"", ""timestamp"": 1680177745605, ""old_value"": 0.115, ""new_value"": 0.12, ""old_abs"": 0.58, ""new_abs"": 0.61}"
5,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,5,5.0,1680177745621,concentration,0.12,0.125,increase,increase_concentration,"{""index"": 5, ""action"": ""concentration"", ""timestamp"": 1680177745621, ""old_value"": 0.12, ""new_value"": 0.125, ""old_abs"": 0.61, ""new_abs"": 0.63}"
6,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,6,6.0,1680177745632,concentration,0.125,0.13,increase,increase_concentration,"{""index"": 6, ""action"": ""concentration"", ""timestamp"": 1680177745632, ""old_value"": 0.125, ""new_value"": 0.13, ""old_abs"": 0.63, ""new_abs"": 0.66}"
7,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,7,7.0,1680177745653,concentration,0.13,0.135,increase,increase_concentration,"{""index"": 7, ""action"": ""concentration"", ""timestamp"": 1680177745653, ""old_value"": 0.13, ""new_value"": 0.135, ""old_abs"": 0.66, ""new_abs"": 0.68}"
8,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,8,8.0,1680177745661,concentration,0.135,0.14,increase,increase_concentration,"{""index"": 8, ""action"": ""concentration"", ""timestamp"": 1680177745661, ""old_value"": 0.135, ""new_value"": 0.14, ""old_abs"": 0.68, ""new_abs"": 0.71}"
9,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,9,9.0,1680177745669,concentration,0.14,0.145,increase,increase_concentration,"{""index"": 9, ""action"": ""concentration"", ""timestamp"": 1680177745669, ""old_value"": 0.14, ""new_value"": 0.145, ""old_abs"": 0.71, ""new_abs"": 0.73}"
10,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,10,10.0,1680177745684,concentration,0.145,0.15,increase,increase_concentration,"{""index"": 10, ""action"": ""concentration"", ""timestamp"": 1680177745684, ""old_value"": 0.145, ""new_value"": 0.15, ""old_abs"": 0.73, ""new_abs"": 0.76}"


action: solution
n events: 680


,file,session_code,subgroup,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,original_entry_json
150,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,150,150.0,1680177759567,solution,drink Mix (red),Cobalt nitrate (red),NaN,solution,"{""index"": 150, ""action"": ""solution"", ""timestamp"": 1680177759567, ""old_value"": ""drink Mix (red)"", ""new_value"": ""Cobalt nitrate (red)"", ""old_wave"": 508, ""new_wave"": 549, ""old_abs"": 0.33, ""new_abs"": 0.47, ""old_conc"": 0.065, ""new_conc"": 0.1}"
162,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,162,162.0,1680177799726,solution,Cobalt nitrate (red),drink Mix (red),NaN,solution,"{""index"": 162, ""action"": ""solution"", ""timestamp"": 1680177799726, ""old_value"": ""Cobalt nitrate (red)"", ""new_value"": ""drink Mix (red)"", ""old_wave"": 549, ""new_wave"": 508, ""old_abs"": 0.47, ""new_abs"": 0.33, ""old_conc"": 0.1, ""new_conc"": 0.065}"
257,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,257,257.0,1680177826848,solution,drink Mix (red),Cobalt nitrate (red),NaN,solution,"{""index"": 257, ""action"": ""solution"", ""timestamp"": 1680177826848, ""old_value"": ""drink Mix (red)"", ""new_value"": ""Cobalt nitrate (red)"", ""old_wave"": 508, ""new_wave"": 549, ""old_abs"": 0.51, ""new_abs"": 0.47}"
266,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,266,266.0,1680177836827,solution,Cobalt nitrate (red),Cobalt chloride (red),NaN,solution,"{""index"": 266, ""action"": ""solution"", ""timestamp"": 1680177836827, ""old_value"": ""Cobalt nitrate (red)"", ""new_value"": ""Cobalt chloride (red)"", ""old_abs"": 0.47, ""new_abs"": 0.72}"
275,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,275,275.0,1680177845590,solution,Cobalt chloride (red),Potassium dichromate (orange),NaN,solution,"{""index"": 275, ""action"": ""solution"", ""timestamp"": 1680177845590, ""old_value"": ""Cobalt chloride (red)"", ""new_value"": ""Potassium dichromate (orange)"", ""old_wave"": 549, ""new_wave"": 392, ""old_abs"": 0.72, ""new_abs"": 0.37, ""old_conc"": 0.1, ""new_conc"": 0.0001}"
284,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,284,284.0,1680177859969,solution,Potassium dichromate (orange),Potassium chromate (yellow),NaN,solution,"{""index"": 284, ""action"": ""solution"", ""timestamp"": 1680177859969, ""old_value"": ""Potassium dichromate (orange)"", ""new_value"": ""Potassium chromate (yellow)"", ""old_wave"": 392, ""new_wave"": 411, ""old_abs"": 0.37, ""new_abs"": 0.48}"
293,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,293,293.0,1680177867633,solution,Potassium chromate (yellow),Nickel chloride (green),NaN,solution,"{""index"": 293, ""action"": ""solution"", ""timestamp"": 1680177867633, ""old_value"": ""Potassium chromate (yellow)"", ""new_value"": ""Nickel chloride (green)"", ""old_wave"": 411, ""new_wave"": 433, ""old_abs"": 0.48, ""new_abs"": 0.53, ""old_conc"": 0.0001, ""new_conc"": 0.1}"
302,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,302,302.0,1680177877384,solution,Nickel chloride (green),Copper sulfate (blue),NaN,solution,"{""index"": 302, ""action"": ""solution"", ""timestamp"": 1680177877384, ""old_value"": ""Nickel chloride (green)"", ""new_value"": ""Copper sulfate (blue)"", ""old_wave"": 433, ""new_wave"": 780, ""old_abs"": 0.53, ""new_abs"": 0.96}"
311,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,311,311.0,1680177886432,solution,Copper sulfate (blue),Potassium permanganate (purple),NaN,solution,"{""index"": 311, ""action"": ""solution"", ""timestamp"": 1680177886432, ""old_value"": ""Copper sulfate (blue)"", ""new_value"": ""Potassium permanganate (purple)"", ""old_wave"": 780, ""new_wave"": 544, ""old_abs"": 0.96, ""new_abs"": 0.2, ""old_conc"": 0.1, ""new_conc"": 0.0001}"
322,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,322,322.0,1680177932010,solution,Potassium permanganate (purple),Nickel chloride (green),NaN,solution,"{""index"": 322, ""action"": ""solution"", ""timestamp"": 1680177932010, ""old_value"": ""Potassium permanganate (purple)"", ""new_value"": ""Nickel chloride (green)"", ""old_wave"": 544, ""n

action: width
n events: 28894


,file,session_code,subgroup,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,original_entry_json
348,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,348,348.0,1680177976572,width,1,1.008,increase,increase_width,"{""index"": 348, ""action"": ""width"", ""timestamp"": 1680177976572, ""old_value"": 1, ""new_value"": 1.008, ""old_abs"": 0.2, ""new_abs"": 0.2}"
349,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,349,349.0,1680177976585,width,1.008,1.016,increase,increase_width,"{""index"": 349, ""action"": ""width"", ""timestamp"": 1680177976585, ""old_value"": 1.008, ""new_value"": 1.016, ""old_abs"": 0.2, ""new_abs"": 0.2}"
350,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,350,350.0,1680177976598,width,1.016,1.024,increase,increase_width,"{""index"": 350, ""action"": ""width"", ""timestamp"": 1680177976598, ""old_value"": 1.016, ""new_value"": 1.024, ""old_abs"": 0.2, ""new_abs"": 0.2}"
351,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,351,351.0,1680177976605,width,1.024,1.032,increase,increase_width,"{""index"": 351, ""action"": ""width"", ""timestamp"": 1680177976605, ""old_value"": 1.024, ""new_value"": 1.032, ""old_abs"": 0.2, ""new_abs"": 0.21}"
352,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,352,352.0,1680177976615,width,1.032,1.04,increase,increase_width,"{""index"": 352, ""action"": ""width"", ""timestamp"": 1680177976615, ""old_value"": 1.032, ""new_value"": 1.04, ""old_abs"": 0.21, ""new_abs"": 0.21}"
353,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,353,353.0,1680177976631,width,1.04,1.056,increase,increase_width,"{""index"": 353, ""action"": ""width"", ""timestamp"": 1680177976631, ""old_value"": 1.04, ""new_value"": 1.056, ""old_abs"": 0.21, ""new_abs"": 0.21}"
354,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,354,354.0,1680177976635,width,1.056,1.064,increase,increase_width,"{""index"": 354, ""action"": ""width"", ""timestamp"": 1680177976635, ""old_value"": 1.056, ""new_value"": 1.064, ""old_abs"": 0.21, ""new_abs"": 0.21}"
355,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,355,355.0,1680177976643,width,1.064,1.072,increase,increase_width,"{""index"": 355, ""action"": ""width"", ""timestamp"": 1680177976643, ""old_value"": 1.064, ""new_value"": 1.072, ""old_abs"": 0.21, ""new_abs"": 0.21}"
356,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,356,356.0,1680177976668,width,1.072,1.08,increase,increase_width,"{""index"": 356, ""action"": ""width"", ""timestamp"": 1680177976668, ""old_value"": 1.072, ""new_value"": 1.08, ""old_abs"": 0.21, ""new_abs"": 0.22}"
357,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,357,357.0,1680177976676,width,1.08,1.088,increase,increase_width,"{""index"": 357, ""action"": ""width"", ""timestamp"": 1680177976676, ""old_value"": 1.08, ""new_value"": 1.088, ""old_abs"": 0.22, ""new_abs"": 0.22}"


action: wavelength
n events: 35876


,file,session_code,subgroup,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,original_entry_json
590,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,590,590.0,1680177989841,wavelength,544,545,increase,increase_wavelength,"{""index"": 590, ""action"": ""wavelength"", ""timestamp"": 1680177989841, ""old_value"": 544, ""new_value"": 545, ""old_abs"": 0.21, ""new_abs"": 0.21}"
594,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,594,594.0,1680177990061,wavelength,545,546,increase,increase_wavelength,"{""index"": 594, ""action"": ""wavelength"", ""timestamp"": 1680177990061, ""old_value"": 545, ""new_value"": 546, ""old_abs"": 0.21, ""new_abs"": 0.2}"
598,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,598,598.0,1680177990237,wavelength,546,547,increase,increase_wavelength,"{""index"": 598, ""action"": ""wavelength"", ""timestamp"": 1680177990237, ""old_value"": 546, ""new_value"": 547, ""old_abs"": 0.2, ""new_abs"": 0.2}"
602,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,602,602.0,1680177990373,wavelength,547,548,increase,increase_wavelength,"{""index"": 602, ""action"": ""wavelength"", ""timestamp"": 1680177990373, ""old_value"": 547, ""new_value"": 548, ""old_abs"": 0.2, ""new_abs"": 0.2}"
606,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,606,606.0,1680177990534,wavelength,548,549,increase,increase_wavelength,"{""index"": 606, ""action"": ""wavelength"", ""timestamp"": 1680177990534, ""old_value"": 548, ""new_value"": 549, ""old_abs"": 0.2, ""new_abs"": 0.2}"
610,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,610,610.0,1680177990707,wavelength,549,550,increase,increase_wavelength,"{""index"": 610, ""action"": ""wavelength"", ""timestamp"": 1680177990707, ""old_value"": 549, ""new_value"": 550, ""old_abs"": 0.2, ""new_abs"": 0.19}"
614,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,614,614.0,1680177990861,wavelength,550,551,increase,increase_wavelength,"{""index"": 614, ""action"": ""wavelength"", ""timestamp"": 1680177990861, ""old_value"": 550, ""new_value"": 551, ""old_abs"": 0.19, ""new_abs"": 0.18}"
618,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,618,618.0,1680177991015,wavelength,551,552,increase,increase_wavelength,"{""index"": 618, ""action"": ""wavelength"", ""timestamp"": 1680177991015, ""old_value"": 551, ""new_value"": 552, ""old_abs"": 0.18, ""new_abs"": 0.18}"
622,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,622,622.0,1680177991177,wavelength,552,553,increase,increase_wavelength,"{""index"": 622, ""action"": ""wavelength"", ""timestamp"": 1680177991177, ""old_value"": 552, ""new_value"": 553, ""old_abs"": 0.18, ""new_abs"": 0.17}"
626,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,626,626.0,1680177991377,wavelength,553,554,increase,increase_wavelength,"{""index"": 626, ""action"": ""wavelength"", ""timestamp"": 1680177991377, ""old_value"": 553, ""new_value"": 554, ""old_abs"": 0.17, ""new_abs"": 0.16}"


In [42]:
BEER_VAR_MAP = {
    "wavelength": "wavelength",
    "width": "path_length",
    "concentration": "concentration",
}

BEER_ALLOWED_ACTIONS = {
    "increase_wavelength",
    "decrease_wavelength",
    "increase_path_length",
    "decrease_path_length",
    "increase_concentration",
    "decrease_concentration",
    "finish",
}

def extract_beer_aligned_trace_from_events(logs):
    aligned_trace = []

    for entry in logs:
        if not isinstance(entry, dict):
            continue

        raw_action = entry.get("action")

        if raw_action == "end":
            aligned_trace.append("finish")
            continue

        if raw_action not in BEER_VAR_MAP:
            continue

        direction = get_direction(entry.get("old_value"), entry.get("new_value"))

        if direction not in {"increase", "decrease"}:
            continue

        mapped_var = BEER_VAR_MAP[raw_action]
        label = f"{direction}_{mapped_var}"

        if label in BEER_ALLOWED_ACTIONS:
            aligned_trace.append(label)

    return aligned_trace

In [43]:
beer_clean_records = []

for _, row in group_b_meta_df.iterrows():
    p = Path(row["path"])
    obj = load_pkl(p)
    logs = obj.get("logs", [])

    clean_trace = extract_beer_aligned_trace_from_events(logs)

    beer_clean_records.append({
        "file": p.name,
        "session_code": obj.get("session_code"),
        "subgroup": obj.get("subgroup"),
        "task": obj.get("task"),
        "n_logs": len(logs),
        "n_clean_actions": len(clean_trace),
        "clean_trace": clean_trace,
        "first_50_clean_actions": clean_trace[:50],
    })

beer_clean_df = pd.DataFrame(beer_clean_records)

display(beer_clean_df.head())
display(beer_clean_df["n_clean_actions"].describe())

,file,session_code,subgroup,task,n_logs,n_clean_actions,clean_trace,first_50_clean_actions
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,1546,984,"[increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, decrease_concentration, ...]","[increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration]"
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,398,297,"[increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increas

count      81.000000
mean     1109.580247
std       755.888845
min        60.000000
25%       604.000000
50%       982.000000
75%      1353.000000
max      4033.000000
Name: n_clean_actions, dtype: float64

In [44]:
beer_clean_output_path = OUTPUT_DIR / "groupB_beer_aligned_clean_traces_v2.jsonl"

with open(beer_clean_output_path, "w", encoding="utf-8") as f:
    for record in beer_clean_records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

print(beer_clean_output_path)

D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_clean_traces_v2.jsonl


After extracting all raw traces from Group B pkl files, the numeric actions were wavelength, width, and concentration. These variables match the Beer’s Law environment rather than the concentration environment used in my model experiment. In particular, the model concentration environment uses solute_amount and solution_volume as manipulable variables, while concentration is the target variable. Therefore, the current Group B data can be aligned to the beers_wavelength environment, but not directly to the concentration environment.

In [45]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from collections import Counter

DATA_DIR = Path(r"D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL")
OUTPUT_DIR = DATA_DIR / "processed_traces"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_EVENT_PATH = OUTPUT_DIR / "groupB_raw_event_table.csv"

print("Raw event path:", RAW_EVENT_PATH)
print("Output dir:", OUTPUT_DIR)

Raw event path: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_event_table.csv
Output dir: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces


In [46]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from collections import Counter

DATA_DIR = Path(r"D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL")
OUTPUT_DIR = DATA_DIR / "processed_traces"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_EVENT_PATH = OUTPUT_DIR / "groupB_raw_event_table.csv"

print("Raw event path:", RAW_EVENT_PATH)
print("Output dir:", OUTPUT_DIR)

Raw event path: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_raw_event_table.csv
Output dir: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces


In [47]:
BEER_VAR_MAP = {
    "wavelength": "wavelength",
    "width": "path_length",
    "concentration": "concentration",
}

BEER_ALLOWED_ACTIONS = {
    "increase_wavelength",
    "decrease_wavelength",
    "increase_path_length",
    "decrease_path_length",
    "increase_concentration",
    "decrease_concentration",
    "finish",
}

In [48]:
def safe_float(x):
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None


def get_direction(old_value, new_value):
    old_f = safe_float(old_value)
    new_f = safe_float(new_value)

    if old_f is None or new_f is None:
        return None

    if new_f > old_f:
        return "increase"
    elif new_f < old_f:
        return "decrease"
    else:
        return "same"


def raw_event_to_beer_label(row):
    raw_action = row["raw_action"]

    if raw_action == "end":
        return "finish"

    if raw_action not in BEER_VAR_MAP:
        return None

    direction = get_direction(row.get("old_value"), row.get("new_value"))

    if direction not in {"increase", "decrease"}:
        return None

    mapped_var = BEER_VAR_MAP[raw_action]
    label = f"{direction}_{mapped_var}"

    if label in BEER_ALLOWED_ACTIONS:
        return label

    return None

In [49]:
aligned_event_df = event_df.copy()

aligned_event_df["beer_label"] = aligned_event_df.apply(
    raw_event_to_beer_label,
    axis=1
)

aligned_event_df = aligned_event_df[
    aligned_event_df["beer_label"].notna()
].copy()

aligned_event_df["timestamp"] = pd.to_numeric(
    aligned_event_df["timestamp"],
    errors="coerce"
)

aligned_event_df["event_pos"] = pd.to_numeric(
    aligned_event_df["event_pos"],
    errors="coerce"
)

aligned_event_df = aligned_event_df.sort_values(
    ["file", "event_pos"]
).reset_index(drop=True)

print("Aligned event rows:", len(aligned_event_df))
display(aligned_event_df.head())

display(
    aligned_event_df["beer_label"]
    .value_counts()
    .rename_axis("beer_label")
    .reset_index(name="count")
)

Aligned event rows: 89876


,event_pos,index,timestamp,raw_action,old_value,new_value,direction,directional_label_raw,has_old_new_value,original_entry_json,file,group,subgroup,session_code,task,beer_label
0,1,1.0,1680177745518,concentration,0.1,0.105,increase,increase_concentration,True,"{""index"": 1, ""action"": ""concentration"", ""timestamp"": 1680177745518, ""old_value"": 0.1, ""new_value"": 0.105, ""old_abs"": 0.51, ""new_abs"": 0.53}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,increase_concentration
1,2,2.0,1680177745549,concentration,0.105,0.11,increase,increase_concentration,True,"{""index"": 2, ""action"": ""concentration"", ""timestamp"": 1680177745549, ""old_value"": 0.105, ""new_value"": 0.11, ""old_abs"": 0.53, ""new_abs"": 0.56}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,increase_concentration
2,3,3.0,1680177745575,concentration,0.11,0.115,increase,increase_concentration,True,"{""index"": 3, ""action"": ""concentration"", ""timestamp"": 1680177745575, ""old_value"": 0.11, ""new_value"": 0.115, ""old_abs"": 0.56, ""new_abs"": 0.58}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,increase_concentration
3,4,4.0,1680177745605,concentration,0.115,0.12,increase,increase_concentration,True,"{""index"": 4, ""action"": ""concentration"", ""timestamp"": 1680177745605, ""old_value"": 0.115, ""new_value"": 0.12, ""old_abs"": 0.58, ""new_abs"": 0.61}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,increase_concentration
4,5,5.0,1680177745621,concentration,0.12,0.125,increase,increase_concentration,True,"{""index"": 5, ""action"": ""concentration"", ""timestamp"": 1680177745621, ""old_value"": 0.12, ""new_value"": 0.125, ""old_abs"": 0.61, ""new_abs"": 0.63}",ex_bckp1-B-DC_sim1.pkl,B,DC,ex_bckp1-B-DC,sim1_task,increase_concentration


,beer_label,count
0,increase_wavelength,19048
1,decrease_wavelength,16828
2,increase_path_length,15267
3,increase_concentration,14999
4,decrease_path_length,13627
5,decrease_concentration,10026
6,finish,81


In [50]:
def make_bursts_for_one_trace(one_df, max_gap_ms=750):
    """
    Input:
        one_df: one session/file's beer-aligned event-level dataframe

    Output:
        burst rows and burst trace

    Rule:
        Merge consecutive same beer_label events if:
        1. same label
        2. both are not finish
        3. timestamp gap <= max_gap_ms

        finish is always kept as its own final action.
    """

    one_df = one_df.sort_values("event_pos").reset_index(drop=True)

    bursts = []
    current = None

    for _, row in one_df.iterrows():
        label = row["beer_label"]
        timestamp = row["timestamp"]
        event_pos = row["event_pos"]

        is_finish = label == "finish"

        if is_finish:
            if current is not None:
                bursts.append(current)
                current = None

            bursts.append({
                "burst_label": "finish",
                "raw_action": row["raw_action"],
                "direction": None,
                "start_event_pos": event_pos,
                "end_event_pos": event_pos,
                "start_timestamp": timestamp,
                "end_timestamp": timestamp,
                "duration_ms": 0,
                "n_micro_events": 1,
                "start_old_value": row.get("old_value"),
                "end_new_value": row.get("new_value"),
            })
            continue

        if current is None:
            current = {
                "burst_label": label,
                "raw_action": row["raw_action"],
                "direction": row.get("direction"),
                "start_event_pos": event_pos,
                "end_event_pos": event_pos,
                "start_timestamp": timestamp,
                "end_timestamp": timestamp,
                "duration_ms": 0,
                "n_micro_events": 1,
                "start_old_value": row.get("old_value"),
                "end_new_value": row.get("new_value"),
            }
            continue

        same_label = label == current["burst_label"]

        if pd.notna(timestamp) and pd.notna(current["end_timestamp"]):
            gap_ms = timestamp - current["end_timestamp"]
            close_in_time = gap_ms <= max_gap_ms
        else:
            gap_ms = None
            close_in_time = False

        if same_label and close_in_time:
            current["end_event_pos"] = event_pos
            current["end_timestamp"] = timestamp
            current["duration_ms"] = (
                current["end_timestamp"] - current["start_timestamp"]
                if pd.notna(current["end_timestamp"]) and pd.notna(current["start_timestamp"])
                else None
            )
            current["n_micro_events"] += 1
            current["end_new_value"] = row.get("new_value")
        else:
            bursts.append(current)

            current = {
                "burst_label": label,
                "raw_action": row["raw_action"],
                "direction": row.get("direction"),
                "start_event_pos": event_pos,
                "end_event_pos": event_pos,
                "start_timestamp": timestamp,
                "end_timestamp": timestamp,
                "duration_ms": 0,
                "n_micro_events": 1,
                "start_old_value": row.get("old_value"),
                "end_new_value": row.get("new_value"),
            }

    if current is not None:
        bursts.append(current)

    burst_trace = [b["burst_label"] for b in bursts]

    return bursts, burst_trace

In [52]:
MAX_GAP_MS = 750

v3_records = []
v3_burst_rows = []

for file_name, one_df in aligned_event_df.groupby("file", sort=False):
    one_df = one_df.sort_values("event_pos").reset_index(drop=True)

    metadata = one_df.iloc[0][
        ["file", "session_code", "subgroup", "task"]
    ].to_dict()

    bursts, burst_trace = make_bursts_for_one_trace(
        one_df,
        max_gap_ms=MAX_GAP_MS
    )

    n_micro_actions = len(one_df)
    n_burst_actions = len(burst_trace)

    v3_records.append({
        **metadata,
        "max_gap_ms": MAX_GAP_MS,
        "n_micro_actions_v2": n_micro_actions,
        "n_burst_actions_v3": n_burst_actions,
        "compression_ratio": (
            n_burst_actions / n_micro_actions
            if n_micro_actions > 0 else None
        ),
        "v3_burst_trace": burst_trace,
        "first_50_v3_actions": burst_trace[:50],
    })

    for burst_id, b in enumerate(bursts):
        v3_burst_rows.append({
            **metadata,
            "burst_id": burst_id,
            "max_gap_ms": MAX_GAP_MS,
            **b,
        })

v3_trace_df = pd.DataFrame(v3_records)
v3_burst_df = pd.DataFrame(v3_burst_rows)

print("Number of V3 traces:", len(v3_trace_df))
print("Number of V3 burst rows:", len(v3_burst_df))

display(v3_trace_df.head())
display(v3_burst_df.head())

Number of V3 traces: 81
Number of V3 burst rows: 5468


,file,session_code,subgroup,task,max_gap_ms,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,v3_burst_trace,first_50_v3_actions
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,750,984,79,0.080285,"[increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, decrease_wavelength, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_wavelength, decrease_wavelength, increase_concentration, increase_concentration, decrease_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, finish]","[increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, decrease_wavelength, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength]"
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,750,297,21,0.070707,"[increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_concentration, increase_concentration, finish]","[increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, in

,file,session_code,subgroup,task,burst_id,max_gap_ms,burst_label,raw_action,direction,start_event_pos,end_event_pos,start_timestamp,end_timestamp,duration_ms,n_micro_events,start_old_value,end_new_value
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,0,750,increase_concentration,concentration,increase,1,58,1680177745518,1680177746680,1162,58,0.100,0.400
1,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,1,750,decrease_concentration,concentration,decrease,59,125,1680177747770,1680177749029,1259,67,0.400,0.000
2,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,2,750,increase_concentration,concentration,increase,126,138,1680177749310,1680177749920,610,13,0.000,0.065
3,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,3,750,increase_concentration,concentration,increase,169,189,1680177803836,1680177804947,1111,6,0.065,0.071
4,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,4,750,increase_concentration,concentration,increase,193,193,1680177806062,1680177806062,0,1,0.071,0.072


In [53]:
display(
    v3_trace_df[
        ["n_micro_actions_v2", "n_burst_actions_v3", "compression_ratio"]
    ].describe()
)

display(
    v3_trace_df[
        ["file", "subgroup", "n_micro_actions_v2", "n_burst_actions_v3", "compression_ratio", "first_50_v3_actions"]
    ].head(10)
)

,n_micro_actions_v2,n_burst_actions_v3,compression_ratio
count,81.000000,81.000000,81.000000
mean,1109.580247,67.506173,0.068318
std,755.888845,46.107788,0.032336
min,60.000000,7.000000,0.028152
25%,604.000000,37.000000,0.049057
50%,982.000000,58.000000,0.059979
75%,1353.000000,84.000000,0.075188
max,4033.000000,229.000000,0.227437


,file,subgroup,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,first_50_v3_actions
0,ex_bckp1-B-DC_sim1.pkl,DC,984,79,0.080285,"[increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, decrease_wavelength, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength]"
1,ex_bckp1-B-DI_sim1.pkl,DI,297,21,0.070707,"[increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_concentration, increase_concentration, finish]"
2,ex_bckp2-B-DC_sim1.pkl,DC,1275,55,0.043137,"[increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, decrease_wavelength, decrease_path_length, decrease_path_length, increase_path_length, decrease_path_length, decrease_wavelength, increase_wavelength, decrease_wavelength, decrease_path_length, increase_path_length, increase_path_length, decrease_path_length, decrease_path_length, increase_wavelength, decrease_wavelength, increase_wavelength, increase_concentration, decrease_concentration, increase_concentration, increase_path_length, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration]"
3,ex_bckp3-B-DC_sim1.pkl,DC,665,33,0.049624,"[decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_concentration, finish]"
4,ex_bckp3-B-DI_sim1.pkl,DI,732,43,0.058743,"[increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_path_length, increase_path_length, decrease_path_length, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, increase_conce

In [54]:
v3_action_counts = Counter()

for trace in v3_trace_df["v3_burst_trace"]:
    v3_action_counts.update(trace)

v3_action_count_df = pd.DataFrame(
    v3_action_counts.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

display(v3_action_count_df)

,action,count
0,increase_concentration,1256
4,increase_wavelength,1104
5,decrease_wavelength,1006
1,decrease_concentration,900
2,increase_path_length,591
3,decrease_path_length,530
6,finish,81


In [55]:
def count_finish(trace):
    return sum(1 for x in trace if x == "finish")

v3_trace_df["n_finish"] = v3_trace_df["v3_burst_trace"].apply(count_finish)
v3_trace_df["finish_is_last"] = v3_trace_df["v3_burst_trace"].apply(
    lambda xs: len(xs) > 0 and xs[-1] == "finish"
)

display(v3_trace_df["n_finish"].value_counts(dropna=False))
display(v3_trace_df["finish_is_last"].value_counts(dropna=False))

display(
    v3_trace_df[
        (v3_trace_df["n_finish"] != 1) | (~v3_trace_df["finish_is_last"])
    ][["file", "n_finish", "finish_is_last", "v3_burst_trace"]]
)

n_finish
1    81
Name: count, dtype: int64

finish_is_last
True    81
Name: count, dtype: int64

,file,n_finish,finish_is_last,v3_burst_trace


In [56]:
v3_jsonl_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_traces_v3_gap{MAX_GAP_MS}ms.jsonl"
v3_overview_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_overview_v3_gap{MAX_GAP_MS}ms.csv"
v3_burst_table_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_table_v3_gap{MAX_GAP_MS}ms.csv"
v3_action_counts_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_action_counts_v3_gap{MAX_GAP_MS}ms.csv"

# full traces
with open(v3_jsonl_path, "w", encoding="utf-8") as f:
    for record in v3_records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

# overview
v3_trace_df.to_csv(
    v3_overview_path,
    index=False,
    encoding="utf-8-sig"
)

# burst-level table
v3_burst_df.to_csv(
    v3_burst_table_path,
    index=False,
    encoding="utf-8-sig"
)

# action counts
v3_action_count_df.to_csv(
    v3_action_counts_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved V3 files:")
print(v3_jsonl_path)
print(v3_overview_path)
print(v3_burst_table_path)
print(v3_action_counts_path)

Saved V3 files:
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_traces_v3_gap750ms.jsonl
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_overview_v3_gap750ms.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_table_v3_gap750ms.csv
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_action_counts_v3_gap750ms.csv


In [57]:
def summarize_gap_threshold(max_gap_ms):
    records = []

    for file_name, one_df in aligned_event_df.groupby("file", sort=False):
        bursts, burst_trace = make_bursts_for_one_trace(
            one_df,
            max_gap_ms=max_gap_ms
        )

        records.append({
            "file": file_name,
            "max_gap_ms": max_gap_ms,
            "n_micro_actions": len(one_df),
            "n_burst_actions": len(burst_trace),
            "compression_ratio": len(burst_trace) / len(one_df) if len(one_df) > 0 else None,
        })

    return pd.DataFrame(records)


threshold_summary_list = []

for gap in [250, 500, 750, 1000, 1500, 2000]:
    tmp = summarize_gap_threshold(gap)
    threshold_summary_list.append({
        "max_gap_ms": gap,
        "mean_burst_actions": tmp["n_burst_actions"].mean(),
        "median_burst_actions": tmp["n_burst_actions"].median(),
        "min_burst_actions": tmp["n_burst_actions"].min(),
        "max_burst_actions": tmp["n_burst_actions"].max(),
        "mean_compression_ratio": tmp["compression_ratio"].mean(),
    })

threshold_summary_df = pd.DataFrame(threshold_summary_list)

display(threshold_summary_df)

,max_gap_ms,mean_burst_actions,median_burst_actions,min_burst_actions,max_burst_actions,mean_compression_ratio
0,250,100.962963,79.0,10,364,0.099001
1,500,76.827160,64.0,7,247,0.077201
2,750,67.506173,58.0,7,229,0.068318
3,1000,64.493827,56.0,7,220,0.064985
4,1500,61.802469,53.0,6,214,0.062073
5,2000,60.246914,52.0,6,206,0.060665


The raw aligned traces contain many fine-grained slider events. After applying timestamp-aware burst aggregation with a 750ms threshold, the average sequence length decreased from 1109.6 micro-actions to 67.5 burst-level actions, corresponding to an average compression ratio of 0.068. This indicates that most repeated actions in the raw traces are high-frequency interaction events rather than independent exploration decisions. Sensitivity analysis over different time thresholds showed that the number of bursts stabilizes after around 750ms, supporting 750ms as a reasonable threshold for the main HMM analysis.

In [58]:
display(v3_action_count_df)
display(v3_trace_df["n_burst_actions_v3"].describe())
display(v3_trace_df.sort_values("n_burst_actions_v3", ascending=False).head(10))

,action,count
0,increase_concentration,1256
4,increase_wavelength,1104
5,decrease_wavelength,1006
1,decrease_concentration,900
2,increase_path_length,591
3,decrease_path_length,530
6,finish,81


count     81.000000
mean      67.506173
std       46.107788
min        7.000000
25%       37.000000
50%       58.000000
75%       84.000000
max      229.000000
Name: n_burst_actions_v3, dtype: float64

,file,session_code,subgroup,task,max_gap_ms,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,v3_burst_trace,first_50_v3_actions,n_finish,finish_is_last
35,k1-f5m9gdjy_sim1.pkl,k1-f5m9gdjy,DI,sim1_task,750,4033,229,0.056782,"[decrease_wavelength, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, decrease_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, decrease_concentration, increase_concentration, decrease_concentration, decrease_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_concentration, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_path_length, increase_path_length, increase_path_length, increase_path_length, increase_concentration, increase_concentration, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_path_length, increase_path_length, decrease_path_length, decrease_path_length, increase_path_length, increase_path_length, increase_path_length, decrease_path_length, increase_path_length, ...]","[decrease_wavelength, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, decrease_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, decrease_concentration, increase_concentration, decrease_concentration, decrease_concentration, increase_concentration]",1,True
68,k1-v7mdsvjn_sim1.pkl,k1-v7mdsvjn,DI,sim1_task,750,2235,206,0.092170,"[increase_wavelength, increase_concentration, increase_concentration, decrease_concentration, decrease_path_length, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentra

In [59]:
from pathlib import Path
import json
import pandas as pd
from collections import Counter

# ===== 路径设置 =====
DATA_DIR = Path(r"D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL")
OUTPUT_DIR = DATA_DIR / "processed_traces"

MAX_GAP_MS = 750

v3_jsonl_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_traces_v3_gap{MAX_GAP_MS}ms.jsonl"
v3_burst_table_path = OUTPUT_DIR / f"groupB_beer_aligned_burst_table_v3_gap{MAX_GAP_MS}ms.csv"

report_path = OUTPUT_DIR / f"groupB_v3_check_report_gap{MAX_GAP_MS}ms.xlsx"

print("Reading:", v3_jsonl_path)
print("Reading:", v3_burst_table_path)
print("Report will be saved to:", report_path)

Reading: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_traces_v3_gap750ms.jsonl
Reading: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_beer_aligned_burst_table_v3_gap750ms.csv
Report will be saved to: D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_v3_check_report_gap750ms.xlsx


In [60]:
# ===== 读取 V3 traces =====
v3_records = []

with open(v3_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        v3_records.append(json.loads(line))

v3_trace_df = pd.DataFrame(v3_records)

# 读取 burst-level table
v3_burst_df = pd.read_csv(v3_burst_table_path)

print("Number of V3 traces:", len(v3_trace_df))
print("Number of burst rows:", len(v3_burst_df))

display(v3_trace_df.head())
display(v3_burst_df.head())

Number of V3 traces: 81
Number of burst rows: 5468


,file,session_code,subgroup,task,max_gap_ms,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,v3_burst_trace,first_50_v3_actions
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,750,984,79,0.080285,"[increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, decrease_wavelength, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_wavelength, decrease_wavelength, increase_concentration, increase_concentration, decrease_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, increase_wavelength, finish]","[increase_concentration, decrease_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, increase_concentration, decrease_concentration, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, decrease_path_length, increase_path_length, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_path_length, decrease_path_length, increase_path_length, increase_concentration, decrease_concentration, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, increase_concentration, decrease_wavelength, decrease_wavelength, decrease_wavelength, increase_concentration, decrease_concentration, decrease_concentration, decrease_concentration, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength]"
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,750,297,21,0.070707,"[increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_concentration, increase_concentration, finish]","[increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, decrease_concentration, increase_concentration, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, increase_wavelength, decrease_wavelength, increase_wavelength, decrease_wavelength, in

,file,session_code,subgroup,task,burst_id,max_gap_ms,burst_label,raw_action,direction,start_event_pos,end_event_pos,start_timestamp,end_timestamp,duration_ms,n_micro_events,start_old_value,end_new_value
0,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,0,750,increase_concentration,concentration,increase,1,58,1680177745518,1680177746680,1162,58,0.100,0.400
1,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,1,750,decrease_concentration,concentration,decrease,59,125,1680177747770,1680177749029,1259,67,0.400,0.000
2,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,2,750,increase_concentration,concentration,increase,126,138,1680177749310,1680177749920,610,13,0.000,0.065
3,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,3,750,increase_concentration,concentration,increase,169,189,1680177803836,1680177804947,1111,6,0.065,0.071
4,ex_bckp1-B-DC_sim1.pkl,ex_bckp1-B-DC,DC,sim1_task,4,750,increase_concentration,concentration,increase,193,193,1680177806062,1680177806062,0,1,0.071,0.072


In [61]:
# ===== 基础检查 =====

def count_finish(trace):
    return sum(1 for x in trace if x == "finish")

def finish_is_last(trace):
    return len(trace) > 0 and trace[-1] == "finish"

# trace length
v3_trace_df["n_finish"] = v3_trace_df["v3_burst_trace"].apply(count_finish)
v3_trace_df["finish_is_last"] = v3_trace_df["v3_burst_trace"].apply(finish_is_last)

# 为了写入 Excel，把 list trace 转成字符串
v3_trace_df["v3_burst_trace_str"] = v3_trace_df["v3_burst_trace"].apply(
    lambda xs: " -> ".join(xs)
)

v3_trace_df["first_100_v3_actions"] = v3_trace_df["v3_burst_trace"].apply(
    lambda xs: " -> ".join(xs[:100])
)

# Excel cell 有长度限制，避免太长
v3_trace_df["v3_burst_trace_str_truncated"] = v3_trace_df["v3_burst_trace_str"].apply(
    lambda x: x[:30000] if isinstance(x, str) else x
)

# action counts
v3_action_counts = Counter()
for trace in v3_trace_df["v3_burst_trace"]:
    v3_action_counts.update(trace)

v3_action_count_df = pd.DataFrame(
    v3_action_counts.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

# summary
summary_df = pd.DataFrame([
    {
        "metric": "n_traces",
        "value": len(v3_trace_df),
    },
    {
        "metric": "n_burst_rows",
        "value": len(v3_burst_df),
    },
    {
        "metric": "max_gap_ms",
        "value": MAX_GAP_MS,
    },
    {
        "metric": "mean_micro_actions_v2",
        "value": v3_trace_df["n_micro_actions_v2"].mean(),
    },
    {
        "metric": "median_micro_actions_v2",
        "value": v3_trace_df["n_micro_actions_v2"].median(),
    },
    {
        "metric": "mean_burst_actions_v3",
        "value": v3_trace_df["n_burst_actions_v3"].mean(),
    },
    {
        "metric": "median_burst_actions_v3",
        "value": v3_trace_df["n_burst_actions_v3"].median(),
    },
    {
        "metric": "mean_compression_ratio",
        "value": v3_trace_df["compression_ratio"].mean(),
    },
    {
        "metric": "min_burst_actions_v3",
        "value": v3_trace_df["n_burst_actions_v3"].min(),
    },
    {
        "metric": "max_burst_actions_v3",
        "value": v3_trace_df["n_burst_actions_v3"].max(),
    },
    {
        "metric": "traces_with_one_finish",
        "value": int((v3_trace_df["n_finish"] == 1).sum()),
    },
    {
        "metric": "traces_finish_is_last",
        "value": int(v3_trace_df["finish_is_last"].sum()),
    },
])

display(summary_df)
display(v3_action_count_df)

,metric,value
0,n_traces,81.000000
1,n_burst_rows,5468.000000
2,max_gap_ms,750.000000
3,mean_micro_actions_v2,1109.580247
4,median_micro_actions_v2,982.000000
5,mean_burst_actions_v3,67.506173
6,median_burst_actions_v3,58.000000
7,mean_compression_ratio,0.068318
8,min_burst_actions_v3,7.000000
9,max_burst_actions_v3,229.000000


,action,count
0,increase_concentration,1256
4,increase_wavelength,1104
5,decrease_wavelength,1006
1,decrease_concentration,900
2,increase_path_length,591
3,decrease_path_length,530
6,finish,81


In [62]:
# ===== 长度分布检查 =====

length_describe_df = v3_trace_df[
    ["n_micro_actions_v2", "n_burst_actions_v3", "compression_ratio"]
].describe().reset_index().rename(columns={"index": "stat"})

# 最长 traces
top_longest_traces_df = (
    v3_trace_df
    .sort_values("n_burst_actions_v3", ascending=False)
    .head(15)
    [[
        "file",
        "session_code",
        "subgroup",
        "task",
        "n_micro_actions_v2",
        "n_burst_actions_v3",
        "compression_ratio",
        "n_finish",
        "finish_is_last",
        "first_100_v3_actions",
    ]]
)

# 最短 traces
top_shortest_traces_df = (
    v3_trace_df
    .sort_values("n_burst_actions_v3", ascending=True)
    .head(15)
    [[
        "file",
        "session_code",
        "subgroup",
        "task",
        "n_micro_actions_v2",
        "n_burst_actions_v3",
        "compression_ratio",
        "n_finish",
        "finish_is_last",
        "first_100_v3_actions",
    ]]
)

# finish 异常
finish_problem_df = v3_trace_df[
    (v3_trace_df["n_finish"] != 1) | (~v3_trace_df["finish_is_last"])
][[
    "file",
    "session_code",
    "subgroup",
    "task",
    "n_micro_actions_v2",
    "n_burst_actions_v3",
    "n_finish",
    "finish_is_last",
    "first_100_v3_actions",
]]

display(length_describe_df)
display(top_longest_traces_df)
display(top_shortest_traces_df)
display(finish_problem_df)

,stat,n_micro_actions_v2,n_burst_actions_v3,compression_ratio
0,count,81.000000,81.000000,81.000000
1,mean,1109.580247,67.506173,0.068318
2,std,755.888845,46.107788,0.032336
3,min,60.000000,7.000000,0.028152
4,25%,604.000000,37.000000,0.049057
5,50%,982.000000,58.000000,0.059979
6,75%,1353.000000,84.000000,0.075188
7,max,4033.000000,229.000000,0.227437


,file,session_code,subgroup,task,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,n_finish,finish_is_last,first_100_v3_actions
35,k1-f5m9gdjy_sim1.pkl,k1-f5m9gdjy,DI,sim1_task,4033,229,0.056782,1,True,decrease_wavelength -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> decrease_concentration -> increase_concentration -> decrease_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_concentration -> decrease_concentration -> decrease_concentration -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> increase_concentration -> increase_concentration -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_concentration -> decrease_path_length -> increase_path_length -> decrease_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length
68,k1-v7mdsvjn_sim1.pkl,k1-v7mdsvjn,DI,sim1_task,2235,206,0.092170,1,True,increase_wavelength -> increase_concentration -> increase_concentration -> decrease_concentration -> decrease_path_length -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> decrease_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> decrease_concentration -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_concentration -> dec

,file,session_code,subgroup,task,n_micro_actions_v2,n_burst_actions_v3,compression_ratio,n_finish,finish_is_last,first_100_v3_actions
20,k1-7fg7oyno_sim1.pkl,k1-7fg7oyno,DC,sim1_task,60,7,0.116667,1,True,increase_concentration -> decrease_wavelength -> decrease_wavelength -> decrease_concentration -> decrease_concentration -> increase_concentration -> finish
13,k1-2ry26mia_sim1.pkl,k1-2ry26mia,DI,sim1_task,131,9,0.068702,1,True,decrease_concentration -> decrease_wavelength -> increase_wavelength -> increase_wavelength -> decrease_wavelength -> increase_concentration -> decrease_concentration -> increase_wavelength -> finish
26,k1-8yhczhdd_sim1.pkl,k1-8yhczhdd,DI,sim1_task,118,13,0.110169,1,True,increase_concentration -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> decrease_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_wavelength -> decrease_wavelength -> finish
25,k1-8wnwvt2o_sim1.pkl,k1-8wnwvt2o,DI,sim1_task,153,14,0.091503,1,True,increase_wavelength -> decrease_wavelength -> increase_wavelength -> decrease_wavelength -> decrease_wavelength -> increase_wavelength -> increase_wavelength -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> finish
21,k1-7tn4j5r_sim1.pkl,k1-7tn4j5r,DC,sim1_task,202,17,0.084158,1,True,increase_concentration -> increase_wavelength -> decrease_path_length -> increase_concentration -> increase_wavelength -> decrease_wavelength -> increase_path_length -> decrease_path_length -> decrease_concentration -> increase_concentration -> increase_wavelength -> increase_wavelength -> increase_path_length -> decrease_path_length -> decrease_path_length -> increase_path_length -> finish
12,k1-2gyhzyv_sim1.pkl,k1-2gyhzyv,DI,sim1_task,308,20,0.064935,1,True,increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> increase_wavelength -> increase_wavelength -> decrease_wavelength -> increase_path_length -> increase_wavelength -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> increase_path_length -> decrease_path_length -> finish
1,ex_bckp1-B-DI_sim1.pkl,ex_bckp1-B-DI,DI,sim1_task,297,21,0.070707,1,True,increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> increase_concentration -> increase_concentration -> finish
5,ex_bckp4-B-DC_sim1.pkl,ex_bckp4-B-DC,DC,sim1_task,545,22,0.040367,1,True,increase_concentration -> decrease_concentration -> decrease_concentration -> increase_wavelength -> decrease_wavelength -> increase_wavelength -> decrease_wavelength -> increase_concentration -> decrease_concentration -> decrease_concentration -> decrease_wavelength -> increase_concentration -> increase_concentration -> increase_wavelength -> increase_path_length -> decrease_path_length -> decrease_path_length -> increase_path_length -> increase_path_length -> increase_concentration -> decrease_concentration -> finish
11,k1-2evmd4o_sim1.pkl,k1-2evmd4o,DI,sim1_task,381,23,0.060367,1,True,increase_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concentration -> increase_concentration -> increase_concentration -> decrease_concentration -> decrease_concentration -> increase_wavelength -> increase_concentration -> decrease_concentration -> increase_concentration -> decrease_concen

,file,session_code,subgroup,task,n_micro_actions_v2,n_burst_actions_v3,n_finish,finish_is_last,first_100_v3_actions


In [63]:
# ===== burst-level 额外检查 =====

# 每种 burst action 的 micro-event 数量
burst_micro_event_summary_df = (
    v3_burst_df
    .groupby("burst_label")["n_micro_events"]
    .describe()
    .reset_index()
)

# 最长 burst，即一个 burst 里合并了最多 micro-events 的情况
top_longest_bursts_df = (
    v3_burst_df
    .sort_values("n_micro_events", ascending=False)
    .head(30)
    [[
        "file",
        "session_code",
        "subgroup",
        "task",
        "burst_id",
        "burst_label",
        "n_micro_events",
        "duration_ms",
        "start_event_pos",
        "end_event_pos",
        "start_timestamp",
        "end_timestamp",
        "start_old_value",
        "end_new_value",
    ]]
)

display(burst_micro_event_summary_df)
display(top_longest_bursts_df)

,burst_label,count,mean,std,min,25%,50%,75%,max
0,decrease_concentration,900.0,11.140000,12.800356,1.0,2.0,6.0,17.0,100.0
1,decrease_path_length,530.0,25.711321,26.726325,1.0,5.0,16.0,39.0,154.0
2,decrease_wavelength,1006.0,16.727634,21.608626,1.0,4.0,10.0,22.0,357.0
3,finish,81.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
4,increase_concentration,1256.0,11.941879,14.491398,1.0,3.0,7.0,16.0,155.0
5,increase_path_length,591.0,25.832487,25.177957,1.0,6.0,17.0,38.0,137.0
6,increase_wavelength,1104.0,17.253623,19.351007,1.0,4.0,11.0,23.0,170.0


,file,session_code,subgroup,task,burst_id,burst_label,n_micro_events,duration_ms,start_event_pos,end_event_pos,start_timestamp,end_timestamp,start_old_value,end_new_value
383,k1-25oz4ao2_sim1.pkl,k1-25oz4ao2,DI,sim1_task,67,decrease_wavelength,357,35601,1814,2170,1680251697436,1680251733037,780.000000,423.000000
480,k1-2c6xw4w_sim1.pkl,k1-2c6xw4w,DC,sim1_task,53,increase_wavelength,170,16915,1341,1510,1685531461003,1685531477918,610.000000,780.000000
1729,k1-ey55om7m_sim1.pkl,k1-ey55om7m,DC,sim1_task,29,decrease_wavelength,169,16792,958,1126,1679903981108,1679903997900,549.000000,380.000000
451,k1-2c6xw4w_sim1.pkl,k1-2c6xw4w,DC,sim1_task,24,increase_wavelength,165,17422,616,806,1685531217808,1685531235230,615.000000,780.000000
1799,k1-ey55om7m_sim1.pkl,k1-ey55om7m,DC,sim1_task,99,increase_concentration,155,15408,3236,3390,1679904391101,1679904406509,0.000100,0.000255
481,k1-2c6xw4w_sim1.pkl,k1-2c6xw4w,DC,sim1_task,54,decrease_path_length,154,3983,1514,1667,1685531480400,1685531484383,2.000000,0.500000
2507,k1-g69gsa77_sim1.pkl,k1-g69gsa77,DC,sim1_task,65,increase_wavelength,146,14510,1685,1830,1680523159415,1680523173925,508.000000,654.000000
3393,k1-nyupgx4w_sim1.pkl,k1-nyupgx4w,DC,sim1_task,3,decrease_path_length,144,2250,132,275,1679985069290,1679985071540,2.000000,0.608000
113,ex_bckp2-B-DC_sim1.pkl,ex_bckp2-B-DC,DC,sim1_task,13,increase_path_length,137,3063,620,756,1680251307456,1680251310519,0.500000,1.945926
2701,k1-h68szyda_sim1.pkl,k1-h68szyda,DI,sim1_task,45,increase_path_length,137,2352,620,756,1680264351337,1680264353689,0.500000,1.800000


In [64]:
# ===== 如果 threshold_summary_df 已经存在，也一起写进去 =====
# 如果不存在，就自动跳过

has_threshold_summary = "threshold_summary_df" in globals()

if has_threshold_summary:
    display(threshold_summary_df)
else:
    print("threshold_summary_df not found. It will be skipped.")

,max_gap_ms,mean_burst_actions,median_burst_actions,min_burst_actions,max_burst_actions,mean_compression_ratio
0,250,100.962963,79.0,10,364,0.099001
1,500,76.827160,64.0,7,247,0.077201
2,750,67.506173,58.0,7,229,0.068318
3,1000,64.493827,56.0,7,220,0.064985
4,1500,61.802469,53.0,6,214,0.062073
5,2000,60.246914,52.0,6,206,0.060665


In [70]:
csv_report_dir = OUTPUT_DIR / f"groupB_v3_check_report_gap{MAX_GAP_MS}ms_csv"
csv_report_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(csv_report_dir / "summary.csv", index=False, encoding="utf-8-sig")
length_describe_df.to_csv(csv_report_dir / "length_describe.csv", index=False, encoding="utf-8-sig")
v3_action_count_df.to_csv(csv_report_dir / "action_counts.csv", index=False, encoding="utf-8-sig")

top_longest_traces_df.to_csv(csv_report_dir / "top_longest_traces.csv", index=False, encoding="utf-8-sig")
top_shortest_traces_df.to_csv(csv_report_dir / "top_shortest_traces.csv", index=False, encoding="utf-8-sig")
finish_problem_df.to_csv(csv_report_dir / "finish_problems.csv", index=False, encoding="utf-8-sig")

burst_micro_event_summary_df.to_csv(csv_report_dir / "burst_micro_summary.csv", index=False, encoding="utf-8-sig")
top_longest_bursts_df.to_csv(csv_report_dir / "top_longest_bursts.csv", index=False, encoding="utf-8-sig")

v3_trace_df[[
    "file",
    "session_code",
    "subgroup",
    "task",
    "max_gap_ms",
    "n_micro_actions_v2",
    "n_burst_actions_v3",
    "compression_ratio",
    "n_finish",
    "finish_is_last",
    "first_100_v3_actions",
    "v3_burst_trace_str_truncated",
]].to_csv(csv_report_dir / "trace_overview.csv", index=False, encoding="utf-8-sig")

v3_burst_df.to_csv(csv_report_dir / "burst_table.csv", index=False, encoding="utf-8-sig")

if has_threshold_summary:
    threshold_summary_df.to_csv(csv_report_dir / "threshold_sensitivity.csv", index=False, encoding="utf-8-sig")

print("Saved CSV report folder to:")
print(csv_report_dir)

Saved CSV report folder to:
D:\111\课程\master thesis\VLM behavior\Mid stage\week12\BLL\processed_traces\groupB_v3_check_report_gap750ms_csv


In [71]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from collections import Counter

PROJECT_DIR = Path(r"\\wsl.localhost\Ubuntu\home\lly\projects\project")

INPUT_ROOTS = [
    PROJECT_DIR / "wandb_downloads_student_simulation",
    PROJECT_DIR / "wandb_downloads_student_simulation_5projects",
]

OUTPUT_DIR = PROJECT_DIR / "processed_model_student_traces"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

for p in INPUT_ROOTS:
    print(p)
    print("exists:", p.exists())
    if p.exists():
        print("first children:", list(p.iterdir())[:5])
    print("-" * 80)

PROJECT_DIR: \\wsl.localhost\Ubuntu\home\lly\projects\project
OUTPUT_DIR: \\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces
\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation
exists: True
first children: [WindowsPath('//wsl.localhost/Ubuntu/home/lly/projects/project/wandb_downloads_student_simulation/scientific-exploration-increase-student-simulation')]
--------------------------------------------------------------------------------
\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation_5projects
exists: True
first children: [WindowsPath('//wsl.localhost/Ubuntu/home/lly/projects/project/wandb_downloads_student_simulation_5projects/scientific-exploration-increase-student-simulation-chart-student'), WindowsPath('//wsl.localhost/Ubuntu/home/lly/projects/project/wandb_downloads_student_simulation_5projects/scientific-exploration-increase-student-simulation-simulation-normal'), WindowsPath('//wsl.loc

In [72]:
TARGET_FILES = {
    "interaction_log.json",
    "steps.json",
    "trajectory.json",
    "summary.json",
    "output.log",
}

scan_records = []

for root in INPUT_ROOTS:
    if not root.exists():
        print("Missing root:", root)
        continue

    for d in root.rglob("*"):
        if not d.is_dir():
            continue

        existing_files = {p.name for p in d.iterdir() if p.is_file()}
        matched_files = sorted(existing_files & TARGET_FILES)

        if matched_files:
            scan_records.append({
                "input_root": root.name,
                "run_dir": str(d),
                "relative_run_dir": str(d.relative_to(root)),
                "n_target_files": len(matched_files),
                "has_interaction_log": "interaction_log.json" in existing_files,
                "has_steps": "steps.json" in existing_files,
                "has_trajectory": "trajectory.json" in existing_files,
                "has_summary": "summary.json" in existing_files,
                "has_output_log": "output.log" in existing_files,
                "matched_files": matched_files,
            })

scan_df = pd.DataFrame(scan_records)

print("Candidate run dirs:", len(scan_df))
display(scan_df.head(20))

if len(scan_df) > 0:
    display(
        scan_df[
            [
                "input_root",
                "has_interaction_log",
                "has_steps",
                "has_trajectory",
                "has_summary",
                "has_output_log",
            ]
        ].value_counts().reset_index(name="n_runs")
    )

scan_path = OUTPUT_DIR / "model_student_simulation_file_scan.csv"
scan_df.to_csv(scan_path, index=False, encoding="utf-8-sig")
print("Saved scan:", scan_path)

Candidate run dirs: 264


,input_root,run_dir,relative_run_dir,n_target_files,has_interaction_log,has_steps,has_trajectory,has_summary,has_output_log,matched_files
0,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
1,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
2,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
3,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
4,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
5,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-09_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-09_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
6,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-08_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-08_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
7,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-06_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-06_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
8,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-00_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_bee

,input_root,has_interaction_log,has_steps,has_trajectory,has_summary,has_output_log,n_runs
0,wandb_downloads_student_simulation_5projects,True,True,True,True,False,200
1,wandb_downloads_student_simulation,True,True,True,True,False,40
2,wandb_downloads_student_simulation_5projects,False,False,False,False,True,20
3,wandb_downloads_student_simulation,False,False,False,False,True,4


Saved scan: \\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_file_scan.csv


In [73]:
def read_json_safe(path):
    try:
        if not Path(path).exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return None


def read_text_safe(path, max_chars=3000):
    try:
        if not Path(path).exists():
            return None
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read(max_chars)
    except Exception:
        return None

In [74]:
def find_first_metadata_from_steps(steps):
    if not isinstance(steps, list):
        return {}

    for step in steps:
        if not isinstance(step, dict):
            continue

        for obs_key in ["observation_before", "observation_after"]:
            obs = step.get(obs_key)
            if isinstance(obs, dict):
                metadata = obs.get("metadata")
                if isinstance(metadata, dict):
                    return metadata

    return {}


def infer_run_metadata(run_dir, steps=None, summary=None):
    metadata = find_first_metadata_from_steps(steps)

    run_dir_str = str(run_dir).lower()

    return {
        "run_dir": str(run_dir),
        "run_name_guess": Path(run_dir).name,

        "simulation_type": metadata.get("simulation_type"),
        "observation_kind": metadata.get("observation_kind"),
        "action_mode": metadata.get("action_mode"),
        "naming_mode": metadata.get("naming_mode"),
        "metadata_level": metadata.get("metadata_level"),
        "target_variable": metadata.get("target_variable"),
        "target_variable_internal": metadata.get("target_variable_internal"),

        "finish_reached": summary.get("finish_reached") if isinstance(summary, dict) else None,
        "finish_step_id": summary.get("finish_step_id") if isinstance(summary, dict) else None,
        "num_steps": summary.get("num_steps") if isinstance(summary, dict) else None,
        "forced_finish": summary.get("forced_finish") if isinstance(summary, dict) else None,
        "parse_error": summary.get("parse_error") if isinstance(summary, dict) else None,

        "path_contains_beer": "beer" in run_dir_str,
        "path_contains_concentration": "concentration" in run_dir_str,
        "path_contains_qwen25vl3b": "qwen25vl3b" in run_dir_str or "qwen25_vl_3b" in run_dir_str,
        "path_contains_qwen25vl7b": "qwen25vl7b" in run_dir_str or "qwen25_vl_7b" in run_dir_str,
        "path_contains_qwen35": "qwen35" in run_dir_str or "qwen3" in run_dir_str,
    }

In [75]:
def extract_trace_from_steps_like(steps_like):
    """
    Input:
        steps_like: list of step dicts from steps.json or trajectory.json

    Output:
        trace:
            ["increase_wavelength", ..., "finish"]

        step_rows:
            one row per model step
    """
    trace = []
    step_rows = []

    if not isinstance(steps_like, list):
        return trace, step_rows

    for step in steps_like:
        if not isinstance(step, dict):
            continue

        step_id = step.get("step_id")
        step_type = step.get("step_type")

        label = None
        action_type = None
        variable = None

        if step_type == "finish":
            label = "finish"

        elif step_type == "action":
            parsed_action = step.get("parsed_action")

            if isinstance(parsed_action, dict):
                action_type = parsed_action.get("action_type")
                variable = parsed_action.get("variable")

                if action_type is not None and variable is not None:
                    label = f"{action_type}_{variable}"

        if label is not None:
            trace.append(label)

        step_rows.append({
            "step_id": step_id,
            "step_type": step_type,
            "action_type": action_type,
            "variable": variable,
            "action_label_raw": label,
            "reasoning": step.get("reasoning"),
            "finish_reason": step.get("finish_reason"),
            "done": step.get("done"),
        })

    return trace, step_rows

In [76]:
BEER_ALLOWED_ACTIONS = {
    "increase_wavelength",
    "decrease_wavelength",
    "increase_concentration",
    "decrease_concentration",
    "increase_path_length",
    "decrease_path_length",
    "finish",
}

BEER_VAR_MAP = {
    "wavelength": "wavelength",
    "concentration": "concentration",
    "path_length": "path_length",
    "width": "path_length",
}


def align_trace_to_beer(trace):
    """
    Align model trace to the human Beer's Law action vocabulary.

    Keep only:
    - increase/decrease wavelength
    - increase/decrease concentration
    - increase/decrease path_length
    - finish

    Important:
    finish is kept only as part of the trace, but later we will exclude
    traces that contain no non-finish beer actions.
    """
    aligned = []

    for label in trace:
        if label == "finish":
            aligned.append("finish")
            continue

        if not isinstance(label, str) or "_" not in label:
            continue

        action_type, variable = label.split("_", 1)

        if variable not in BEER_VAR_MAP:
            continue

        mapped_variable = BEER_VAR_MAP[variable]
        mapped_label = f"{action_type}_{mapped_variable}"

        if mapped_label in BEER_ALLOWED_ACTIONS:
            aligned.append(mapped_label)

    return aligned


def count_non_finish_actions(trace):
    return sum(1 for x in trace if x != "finish")

In [77]:
model_trace_records = []
model_step_event_records = []

for _, row in scan_df.iterrows():
    run_dir = Path(row["run_dir"])

    steps_path = run_dir / "steps.json"
    trajectory_path = run_dir / "trajectory.json"
    summary_path = run_dir / "summary.json"
    interaction_log_path = run_dir / "interaction_log.json"
    output_log_path = run_dir / "output.log"

    steps = read_json_safe(steps_path)
    trajectory = read_json_safe(trajectory_path)
    summary = read_json_safe(summary_path)
    interaction_log = read_json_safe(interaction_log_path)
    output_log_head = read_text_safe(output_log_path, max_chars=2000)

    # 优先使用 steps.json；没有则用 trajectory.json
    if isinstance(steps, list):
        raw_trace, step_rows = extract_trace_from_steps_like(steps)
        source_used = "steps.json"
        steps_like_for_metadata = steps

    elif isinstance(trajectory, list):
        raw_trace, step_rows = extract_trace_from_steps_like(trajectory)
        source_used = "trajectory.json"
        steps_like_for_metadata = trajectory

    else:
        raw_trace, step_rows = [], []
        source_used = "none"
        steps_like_for_metadata = None

    beer_trace = align_trace_to_beer(raw_trace)

    meta = infer_run_metadata(
        run_dir=run_dir,
        steps=steps_like_for_metadata,
        summary=summary,
    )

    simulation_type = meta.get("simulation_type")

    if simulation_type is None:
        if meta["path_contains_beer"]:
            simulation_type_guess = "beer_or_beers_wavelength"
        elif meta["path_contains_concentration"]:
            simulation_type_guess = "concentration"
        else:
            simulation_type_guess = "unknown"
    else:
        simulation_type_guess = simulation_type

    n_beer_nonfinish_actions = count_non_finish_actions(beer_trace)

    record = {
        "input_root": row["input_root"],
        "relative_run_dir": row["relative_run_dir"],
        "source_used": source_used,

        **meta,

        "has_steps": steps is not None,
        "has_trajectory": trajectory is not None,
        "has_summary": summary is not None,
        "has_interaction_log": interaction_log is not None,
        "has_output_log": output_log_head is not None,

        "simulation_type_guess": simulation_type_guess,

        "n_raw_trace_actions": len(raw_trace),
        "n_beer_aligned_actions": len(beer_trace),
        "n_beer_nonfinish_actions": n_beer_nonfinish_actions,

        "raw_trace": raw_trace,
        "beer_aligned_trace": beer_trace,

        "first_50_raw_actions": raw_trace[:50],
        "first_50_beer_actions": beer_trace[:50],

        "raw_action_counts": dict(Counter(raw_trace)),
        "beer_action_counts": dict(Counter(beer_trace)),
    }

    model_trace_records.append(record)

    for step_row in step_rows:
        step_row.update({
            "input_root": row["input_root"],
            "relative_run_dir": row["relative_run_dir"],
            "run_dir": str(run_dir),
            "source_used": source_used,
            "simulation_type_guess": simulation_type_guess,
            "observation_kind": meta.get("observation_kind"),
            "action_mode": meta.get("action_mode"),
            "target_variable": meta.get("target_variable"),
        })
        model_step_event_records.append(step_row)

model_trace_df = pd.DataFrame(model_trace_records)
model_step_event_df = pd.DataFrame(model_step_event_records)

print("Number of extracted traces:", len(model_trace_df))
print("Number of step events:", len(model_step_event_df))

display(model_trace_df.head())
display(model_step_event_df.head())

Number of extracted traces: 264
Number of step events: 2975


,input_root,relative_run_dir,source_used,run_dir,run_name_guess,simulation_type,observation_kind,action_mode,naming_mode,metadata_level,...,simulation_type_guess,n_raw_trace_actions,n_beer_aligned_actions,n_beer_nonfinish_actions,raw_trace,beer_aligned_trace,first_50_raw_actions,first_50_beer_actions,raw_action_counts,beer_action_counts
0,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,episode-student_concentration_qwen25vl3b-03_latest,None,None,None,None,None,...,concentration,48,1,0,"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish],"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish],"{'increase_solute_amount': 10, 'increase_solution_volume': 37, 'finish': 1}",{'finish': 1}
1,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,steps.json,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,episode-student_concentration_qwen25vl3b-07_latest,None,None,None,None,None,...,concentration,13,1,0,"[increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, inc

,step_id,step_type,action_type,variable,action_label_raw,reasoning,finish_reason,done,input_root,relative_run_dir,run_dir,source_used,simulation_type_guess,observation_kind,action_mode,target_variable
0,0,action,increase,solute_amount,increase_solute_amount,Increase the solute amount to observe how it affects the concentration.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None
1,1,action,increase,solute_amount,increase_solute_amount,Increasing solute amount will likely affect the concentration.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None
2,2,action,increase,solution_volume,increase_solution_volume,I need to adjust the concentration by increasing the solution volume.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None
3,3,action,increase,solution_volume,increase_solution_volume,The concentration needs to be increased further.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None
4,4,action,increase,solute_amount,increase_solute_amount,Increasing solute amount.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None


In [78]:
display(model_trace_df["source_used"].value_counts(dropna=False))
display(model_trace_df["simulation_type_guess"].value_counts(dropna=False))
display(model_trace_df["observation_kind"].value_counts(dropna=False))
display(model_trace_df["action_mode"].value_counts(dropna=False))

display(
    model_trace_df[
        [
            "input_root",
            "relative_run_dir",
            "source_used",
            "simulation_type_guess",
            "observation_kind",
            "action_mode",
            "target_variable",
            "n_raw_trace_actions",
            "n_beer_aligned_actions",
            "n_beer_nonfinish_actions",
            "finish_reached",
            "finish_step_id",
            "num_steps",
            "forced_finish",
            "parse_error",
            "first_50_raw_actions",
            "first_50_beer_actions",
        ]
    ].head(30)
)

source_used
steps.json    240
none           24
Name: count, dtype: int64

simulation_type_guess
concentration               132
beer_or_beers_wavelength    132
Name: count, dtype: int64

observation_kind
None    264
Name: count, dtype: int64

action_mode
None    264
Name: count, dtype: int64

,input_root,relative_run_dir,source_used,simulation_type_guess,observation_kind,action_mode,target_variable,n_raw_trace_actions,n_beer_aligned_actions,n_beer_nonfinish_actions,finish_reached,finish_step_id,num_steps,forced_finish,parse_error,first_50_raw_actions,first_50_beer_actions
0,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,steps.json,concentration,None,None,None,48,1,0,True,47.0,48.0,False,None,"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish]
1,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,steps.json,concentration,None,None,None,13,1,0,True,12.0,13.0,False,None,"[increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, finish]",[finish]
2,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,steps.json,concentration,None,None,None,3,1,0,True,2.0,3.0,False,None,"[increase_solute_amount, increase_solution_volume, finish]",[finish]
3,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,steps.json,concentration,None,None,None,27,1,0,True,26.0,27.0,False,None,"[increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish]
4,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,steps.json,concentration,None,None,None,18,1,0,True,17.0,18.0,False,None,"[increase_solute_amount, increase_solute_amount, decrease_solute_amount, decrease_solute_amount, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volu

In [79]:
all_raw_model_actions = Counter()
all_beer_model_actions = Counter()

for xs in model_trace_df["raw_trace"]:
    all_raw_model_actions.update(xs)

for xs in model_trace_df["beer_aligned_trace"]:
    all_beer_model_actions.update(xs)

raw_model_action_count_df = pd.DataFrame(
    all_raw_model_actions.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

beer_model_action_count_df = pd.DataFrame(
    all_beer_model_actions.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

display(raw_model_action_count_df)
display(beer_model_action_count_df)

,action,count
0,increase_solute_amount,825
5,increase_wavelength,508
1,increase_solution_volume,464
2,finish,226
6,increase_concentration,178
3,decrease_solute_amount,177
4,decrease_solution_volume,156
9,decrease_wavelength,156
7,increase_path_length,141
10,decrease_concentration,87


,action,count
1,increase_wavelength,508
0,finish,226
2,increase_concentration,178
5,decrease_wavelength,156
3,increase_path_length,141
6,decrease_concentration,87
4,decrease_path_length,51


In [80]:
steps_scan_df = scan_df[scan_df["has_steps"] == True].copy()

print("Runs with steps.json:", len(steps_scan_df))
display(steps_scan_df.head())

display(steps_scan_df["input_root"].value_counts())

Runs with steps.json: 240


,input_root,run_dir,relative_run_dir,n_target_files,has_interaction_log,has_steps,has_trajectory,has_summary,has_output_log,matched_files
0,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
1,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
2,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
3,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"
4,wandb_downloads_student_simulation,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,4,True,True,True,True,False,"[interaction_log.json, steps.json, summary.json, trajectory.json]"


input_root
wandb_downloads_student_simulation_5projects    200
wandb_downloads_student_simulation               40
Name: count, dtype: int64

In [81]:
def extract_trace_from_steps(steps):
    """
    Extract one model/student-simulation trace from steps.json.

    Output trace example:
    [
        "increase_wavelength",
        "increase_path_length",
        "decrease_concentration",
        "finish"
    ]
    """
    trace = []
    step_rows = []

    if not isinstance(steps, list):
        return trace, step_rows

    for step in steps:
        step_id = step.get("step_id")
        step_type = step.get("step_type")

        label = None
        action_type = None
        variable = None

        if step_type == "finish":
            label = "finish"

        elif step_type == "action":
            parsed_action = step.get("parsed_action")

            if isinstance(parsed_action, dict):
                action_type = parsed_action.get("action_type")
                variable = parsed_action.get("variable")

                if action_type is not None and variable is not None:
                    label = f"{action_type}_{variable}"

        if label is not None:
            trace.append(label)

        step_rows.append({
            "step_id": step_id,
            "step_type": step_type,
            "action_type": action_type,
            "variable": variable,
            "action_label": label,
            "reasoning": step.get("reasoning"),
            "finish_reason": step.get("finish_reason"),
            "done": step.get("done"),
        })

    return trace, step_rows

In [82]:
BEER_ALLOWED_ACTIONS = {
    "increase_wavelength",
    "decrease_wavelength",
    "increase_concentration",
    "decrease_concentration",
    "increase_path_length",
    "decrease_path_length",
    "finish",
}

BEER_VAR_MAP = {
    "wavelength": "wavelength",
    "concentration": "concentration",
    "path_length": "path_length",
    "width": "path_length",
}


def align_trace_to_beer(trace):
    aligned = []

    for label in trace:
        if label == "finish":
            aligned.append("finish")
            continue

        if not isinstance(label, str) or "_" not in label:
            continue

        action_type, variable = label.split("_", 1)

        if variable not in BEER_VAR_MAP:
            continue

        mapped_variable = BEER_VAR_MAP[variable]
        mapped_label = f"{action_type}_{mapped_variable}"

        if mapped_label in BEER_ALLOWED_ACTIONS:
            aligned.append(mapped_label)

    return aligned


def count_non_finish(trace):
    return sum(1 for x in trace if x != "finish")

In [83]:
def infer_simulation_type_from_path(relative_run_dir):
    s = str(relative_run_dir).lower()

    if "beer" in s:
        return "beer"
    elif "concentration" in s:
        return "concentration"
    else:
        return "unknown"


def trace_to_str(trace):
    return " -> ".join(trace) if isinstance(trace, list) else ""

In [84]:
model_trace_records = []
model_step_event_records = []

for _, row in steps_scan_df.iterrows():
    run_dir = Path(row["run_dir"])
    steps_path = run_dir / "steps.json"
    summary_path = run_dir / "summary.json"

    steps = read_json_safe(steps_path)
    summary = read_json_safe(summary_path)

    raw_trace, step_rows = extract_trace_from_steps(steps)
    beer_aligned_trace = align_trace_to_beer(raw_trace)

    simulation_type_guess = infer_simulation_type_from_path(row["relative_run_dir"])

    n_raw_actions = len(raw_trace)
    n_beer_actions = len(beer_aligned_trace)
    n_beer_nonfinish = count_non_finish(beer_aligned_trace)

    record = {
        "input_root": row["input_root"],
        "relative_run_dir": row["relative_run_dir"],
        "run_dir": str(run_dir),
        "simulation_type_guess": simulation_type_guess,

        "n_raw_actions": n_raw_actions,
        "n_beer_aligned_actions": n_beer_actions,
        "n_beer_nonfinish_actions": n_beer_nonfinish,

        "raw_trace": raw_trace,
        "beer_aligned_trace": beer_aligned_trace,

        "finish_reached": summary.get("finish_reached") if isinstance(summary, dict) else None,
        "finish_step_id": summary.get("finish_step_id") if isinstance(summary, dict) else None,
        "num_steps": summary.get("num_steps") if isinstance(summary, dict) else None,
        "forced_finish": summary.get("forced_finish") if isinstance(summary, dict) else None,
        "parse_error": summary.get("parse_error") if isinstance(summary, dict) else None,

        "raw_trace_str": trace_to_str(raw_trace),
        "beer_aligned_trace_str": trace_to_str(beer_aligned_trace),
        "first_50_raw_actions": raw_trace[:50],
        "first_50_beer_actions": beer_aligned_trace[:50],
    }

    model_trace_records.append(record)

    for step_row in step_rows:
        step_row.update({
            "input_root": row["input_root"],
            "relative_run_dir": row["relative_run_dir"],
            "run_dir": str(run_dir),
            "simulation_type_guess": simulation_type_guess,
        })
        model_step_event_records.append(step_row)

model_trace_df = pd.DataFrame(model_trace_records)
model_step_event_df = pd.DataFrame(model_step_event_records)

print("Total standard traces:", len(model_trace_df))
print("Total step events:", len(model_step_event_df))

display(model_trace_df.head())
display(model_step_event_df.head())

Total standard traces: 240
Total step events: 2975


,input_root,relative_run_dir,run_dir,simulation_type_guess,n_raw_actions,n_beer_aligned_actions,n_beer_nonfinish_actions,raw_trace,beer_aligned_trace,finish_reached,finish_step_id,num_steps,forced_finish,parse_error,raw_trace_str,beer_aligned_trace_str,first_50_raw_actions,first_50_beer_actions
0,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration,48,1,0,"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish],True,47,48,False,None,increase_solute_amount -> increase_solute_amount -> increase_solution_volume -> increase_solution_volume -> increase_solute_amount -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solute_amount -> increase_solute_amount -> increase_solute_amount -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solute_amount -> increase_solute_amount -> increase_solute_amount -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solute_amount -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> increase_solution_volume -> finish,finish,"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volu

,step_id,step_type,action_type,variable,action_label,reasoning,finish_reason,done,input_root,relative_run_dir,run_dir,simulation_type_guess
0,0,action,increase,solute_amount,increase_solute_amount,Increase the solute amount to observe how it affects the concentration.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration
1,1,action,increase,solute_amount,increase_solute_amount,Increasing solute amount will likely affect the concentration.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration
2,2,action,increase,solution_volume,increase_solution_volume,I need to adjust the concentration by increasing the solution volume.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration
3,3,action,increase,solution_volume,increase_solution_volume,The concentration needs to be increased further.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration
4,4,action,increase,solute_amount,increase_solute_amount,Increasing solute amount.,NaN,False,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,\\wsl.localhost\Ubuntu\home\lly\projects\project\wandb_downloads_student_simulation\scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration


In [85]:
display(model_trace_df["simulation_type_guess"].value_counts(dropna=False))

display(
    model_trace_df[
        [
            "input_root",
            "relative_run_dir",
            "simulation_type_guess",
            "n_raw_actions",
            "n_beer_aligned_actions",
            "n_beer_nonfinish_actions",
            "finish_reached",
            "finish_step_id",
            "num_steps",
            "forced_finish",
            "parse_error",
            "first_50_raw_actions",
            "first_50_beer_actions",
        ]
    ].head(30)
)

simulation_type_guess
concentration    120
beer             120
Name: count, dtype: int64

,input_root,relative_run_dir,simulation_type_guess,n_raw_actions,n_beer_aligned_actions,n_beer_nonfinish_actions,finish_reached,finish_step_id,num_steps,forced_finish,parse_error,first_50_raw_actions,first_50_beer_actions
0,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-03_latest,concentration,48,1,0,True,47,48,False,None,"[increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish]
1,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-07_latest,concentration,13,1,0,True,12,13,False,None,"[increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solute_amount, finish]",[finish]
2,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl7b-06_latest,concentration,3,1,0,True,2,3,False,None,"[increase_solute_amount, increase_solution_volume, finish]",[finish]
3,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-08_latest,concentration,27,1,0,True,26,27,False,None,"[increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solute_amount, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, increase_solution_volume, finish]",[finish]
4,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_concentration_qwen25vl3b-05_latest,concentration,18,1,0,True,17,18,False,None,"[increase_solute_amount, increase_solute_amount, decrease_solute_amount, decrease_solute_amount, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, decrease_solution_volume, finish]",[finish]
5,wandb_downloads_student_simulation,scientific-exploration-i

In [86]:
beer_model_df = model_trace_df[
    model_trace_df["n_beer_nonfinish_actions"] > 0
].copy()

beer_model_df["n_finish"] = beer_model_df["beer_aligned_trace"].apply(
    lambda xs: sum(1 for x in xs if x == "finish")
)

beer_model_df["finish_is_last"] = beer_model_df["beer_aligned_trace"].apply(
    lambda xs: len(xs) > 0 and xs[-1] == "finish"
)

display(
    beer_model_df[
        ["n_finish", "finish_is_last"]
    ].value_counts().reset_index(name="n_runs")
)

beer_hmm_ready_df = beer_model_df[
    (beer_model_df["n_finish"] == 1)
    & (beer_model_df["finish_is_last"])
].copy()

print("Beer HMM-ready traces:", len(beer_hmm_ready_df))

display(
    beer_hmm_ready_df[
        [
            "input_root",
            "relative_run_dir",
            "simulation_type_guess",
            "n_beer_aligned_actions",
            "n_beer_nonfinish_actions",
            "finish_reached",
            "finish_step_id",
            "num_steps",
            "forced_finish",
            "parse_error",
            "beer_aligned_trace_str",
        ]
    ].head(30)
)

,n_finish,finish_is_last,n_runs
0,1,True,117
1,0,False,3


Beer HMM-ready traces: 117


,input_root,relative_run_dir,simulation_type_guess,n_beer_aligned_actions,n_beer_nonfinish_actions,finish_reached,finish_step_id,num_steps,forced_finish,parse_error,beer_aligned_trace_str
5,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-09_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> increase_wavelength -> increase_wavelength -> increase_concentration -> finish
6,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-08_latest,beer,5,4,True,4,5,False,None,increase_concentration -> increase_path_length -> increase_path_length -> decrease_path_length -> finish
7,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-06_latest,beer,4,3,True,3,4,False,None,increase_concentration -> increase_path_length -> increase_wavelength -> finish
8,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-00_latest,beer,6,5,True,5,6,False,None,increase_wavelength -> increase_path_length -> increase_path_length -> increase_path_length -> increase_concentration -> finish
9,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-06_latest,beer,4,3,True,3,4,False,None,increase_wavelength -> decrease_wavelength -> increase_concentration -> finish
10,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-07_latest,beer,3,2,True,2,3,False,None,increase_wavelength -> increase_wavelength -> finish
13,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-01_latest,beer,7,6,True,6,7,False,None,increase_wavelength -> decrease_wavelength -> decrease_wavelength -> increase_wavelength -> increase_concentration -> increase_path_length -> finish
14,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-08_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> increase_concentration -> increase_path_length -> increase_concentration -> finish
16,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-05_latest,beer,2,1,True,1,2,False,None,increase_wavelength -> finish
20,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-01_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> decrease_path_length -> decrease_wavelength -> increase_wavelength -> finish


In [87]:
raw_action_counts = Counter()
beer_action_counts = Counter()

for trace in model_trace_df["raw_trace"]:
    raw_action_counts.update(trace)

for trace in beer_hmm_ready_df["beer_aligned_trace"]:
    beer_action_counts.update(trace)

raw_action_count_df = pd.DataFrame(
    raw_action_counts.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

beer_action_count_df = pd.DataFrame(
    beer_action_counts.items(),
    columns=["action", "count"]
).sort_values("count", ascending=False)

display(raw_action_count_df)
display(beer_action_count_df)

,action,count
0,increase_solute_amount,825
5,increase_wavelength,508
1,increase_solution_volume,464
2,finish,226
6,increase_concentration,178
3,decrease_solute_amount,177
4,decrease_solution_volume,156
9,decrease_wavelength,156
7,increase_path_length,141
10,decrease_concentration,87


,action,count
0,increase_wavelength,422
1,increase_concentration,163
5,decrease_wavelength,140
2,finish,117
3,increase_path_length,113
6,decrease_concentration,82
4,decrease_path_length,51


In [88]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 所有 240 条标准 traces
all_jsonl_path = OUTPUT_DIR / "model_student_simulation_240_standard_traces.jsonl"
all_overview_path = OUTPUT_DIR / "model_student_simulation_240_standard_trace_overview.csv"

with open(all_jsonl_path, "w", encoding="utf-8") as f:
    for record in model_trace_records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

model_trace_df[
    [
        "input_root",
        "relative_run_dir",
        "run_dir",
        "simulation_type_guess",
        "n_raw_actions",
        "n_beer_aligned_actions",
        "n_beer_nonfinish_actions",
        "finish_reached",
        "finish_step_id",
        "num_steps",
        "forced_finish",
        "parse_error",
        "raw_trace_str",
        "beer_aligned_trace_str",
    ]
].to_csv(all_overview_path, index=False, encoding="utf-8-sig")

# 2. step-level table
step_event_path = OUTPUT_DIR / "model_student_simulation_240_step_event_table.csv"
model_step_event_df.to_csv(step_event_path, index=False, encoding="utf-8-sig")

# 3. beer HMM-ready traces
beer_hmm_jsonl_path = OUTPUT_DIR / "model_student_simulation_beer_hmm_ready_traces.jsonl"
beer_hmm_overview_path = OUTPUT_DIR / "model_student_simulation_beer_hmm_ready_overview.csv"

with open(beer_hmm_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in beer_hmm_ready_df.iterrows():
        out = {
            "input_root": row["input_root"],
            "relative_run_dir": row["relative_run_dir"],
            "run_dir": row["run_dir"],
            "simulation_type_guess": row["simulation_type_guess"],
            "n_actions": row["n_beer_aligned_actions"],
            "n_nonfinish_actions": row["n_beer_nonfinish_actions"],
            "trace": row["beer_aligned_trace"],
            "finish_reached": row["finish_reached"],
            "forced_finish": row["forced_finish"],
            "parse_error": row["parse_error"],
        }
        f.write(json.dumps(out, ensure_ascii=False, default=str) + "\n")

beer_hmm_ready_df[
    [
        "input_root",
        "relative_run_dir",
        "run_dir",
        "simulation_type_guess",
        "n_beer_aligned_actions",
        "n_beer_nonfinish_actions",
        "finish_reached",
        "finish_step_id",
        "num_steps",
        "forced_finish",
        "parse_error",
        "beer_aligned_trace_str",
    ]
].to_csv(beer_hmm_overview_path, index=False, encoding="utf-8-sig")

# 4. action count
raw_action_count_path = OUTPUT_DIR / "model_student_simulation_240_raw_action_counts.csv"
beer_action_count_path = OUTPUT_DIR / "model_student_simulation_beer_hmm_ready_action_counts.csv"

raw_action_count_df.to_csv(raw_action_count_path, index=False, encoding="utf-8-sig")
beer_action_count_df.to_csv(beer_action_count_path, index=False, encoding="utf-8-sig")

print("Saved files:")
print(all_jsonl_path)
print(all_overview_path)
print(step_event_path)
print(beer_hmm_jsonl_path)
print(beer_hmm_overview_path)
print(raw_action_count_path)
print(beer_action_count_path)

Saved files:
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_240_standard_traces.jsonl
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_240_standard_trace_overview.csv
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_240_step_event_table.csv
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_beer_hmm_ready_traces.jsonl
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_beer_hmm_ready_overview.csv
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_240_raw_action_counts.csv
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_beer_hmm_ready_action_counts.csv


In [89]:
beer_model_df = model_trace_df[
    (model_trace_df["simulation_type_guess"] == "beer")
    & (model_trace_df["n_beer_nonfinish_actions"] > 0)
].copy()

beer_model_df["n_finish"] = beer_model_df["beer_aligned_trace"].apply(
    lambda xs: sum(1 for x in xs if x == "finish")
)

beer_model_df["finish_is_last"] = beer_model_df["beer_aligned_trace"].apply(
    lambda xs: len(xs) > 0 and xs[-1] == "finish"
)

beer_hmm_ready_df = beer_model_df[
    (beer_model_df["n_finish"] == 1)
    & (beer_model_df["finish_is_last"])
].copy()

print("Beer HMM-ready traces:", len(beer_hmm_ready_df))

display(
    beer_hmm_ready_df[
        [
            "input_root",
            "relative_run_dir",
            "simulation_type_guess",
            "n_beer_aligned_actions",
            "n_beer_nonfinish_actions",
            "finish_reached",
            "finish_step_id",
            "num_steps",
            "forced_finish",
            "parse_error",
            "beer_aligned_trace_str",
        ]
    ].head(20)
)

Beer HMM-ready traces: 116


,input_root,relative_run_dir,simulation_type_guess,n_beer_aligned_actions,n_beer_nonfinish_actions,finish_reached,finish_step_id,num_steps,forced_finish,parse_error,beer_aligned_trace_str
5,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-09_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> increase_wavelength -> increase_wavelength -> increase_concentration -> finish
6,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-08_latest,beer,5,4,True,4,5,False,None,increase_concentration -> increase_path_length -> increase_path_length -> decrease_path_length -> finish
7,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-06_latest,beer,4,3,True,3,4,False,None,increase_concentration -> increase_path_length -> increase_wavelength -> finish
8,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-00_latest,beer,6,5,True,5,6,False,None,increase_wavelength -> increase_path_length -> increase_path_length -> increase_path_length -> increase_concentration -> finish
9,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-06_latest,beer,4,3,True,3,4,False,None,increase_wavelength -> decrease_wavelength -> increase_concentration -> finish
10,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-07_latest,beer,3,2,True,2,3,False,None,increase_wavelength -> increase_wavelength -> finish
13,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl7b-01_latest,beer,7,6,True,6,7,False,None,increase_wavelength -> decrease_wavelength -> decrease_wavelength -> increase_wavelength -> increase_concentration -> increase_path_length -> finish
14,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-08_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> increase_concentration -> increase_path_length -> increase_concentration -> finish
16,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-05_latest,beer,2,1,True,1,2,False,None,increase_wavelength -> finish
20,wandb_downloads_student_simulation,scientific-exploration-increase-student-simulation\artifacts\episode-student_beers_wavelength_qwen25vl3b-01_latest,beer,5,4,True,4,5,False,None,increase_wavelength -> decrease_path_length -> decrease_wavelength -> increase_wavelength -> finish


In [90]:
beer_hmm_jsonl_path = OUTPUT_DIR / "model_student_simulation_beer_hmm_ready_traces_corrected.jsonl"
beer_hmm_overview_path = OUTPUT_DIR / "model_student_simulation_beer_hmm_ready_overview_corrected.csv"

with open(beer_hmm_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in beer_hmm_ready_df.iterrows():
        out = {
            "input_root": row["input_root"],
            "relative_run_dir": row["relative_run_dir"],
            "run_dir": row["run_dir"],
            "simulation_type_guess": row["simulation_type_guess"],
            "n_actions": row["n_beer_aligned_actions"],
            "n_nonfinish_actions": row["n_beer_nonfinish_actions"],
            "trace": row["beer_aligned_trace"],
            "finish_reached": row["finish_reached"],
            "forced_finish": row["forced_finish"],
            "parse_error": row["parse_error"],
        }
        f.write(json.dumps(out, ensure_ascii=False, default=str) + "\n")

beer_hmm_ready_df[
    [
        "input_root",
        "relative_run_dir",
        "run_dir",
        "simulation_type_guess",
        "n_beer_aligned_actions",
        "n_beer_nonfinish_actions",
        "finish_reached",
        "finish_step_id",
        "num_steps",
        "forced_finish",
        "parse_error",
        "beer_aligned_trace_str",
    ]
].to_csv(beer_hmm_overview_path, index=False, encoding="utf-8-sig")

print("Saved corrected files:")
print(beer_hmm_jsonl_path)
print(beer_hmm_overview_path)

Saved corrected files:
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_beer_hmm_ready_traces_corrected.jsonl
\\wsl.localhost\Ubuntu\home\lly\projects\project\processed_model_student_traces\model_student_simulation_beer_hmm_ready_overview_corrected.csv


In [92]:
%pip install -U pip setuptools wheel

Note: you may need to restart the kernel to use updated packages.


�û��������벻��ȷ��
